# Quito Road Model — Interactive Scenarios

This notebook builds a **macroscopic road-traffic model of Quito, Ecuador**, and then runs
*what-if* scenarios on it.

First, the baseline model:

1. Download the road network from OpenStreetMap.
2. Define Traffic Analysis Zones (TAZs) and connect them to the network.
3. Populate link attributes (speed, capacity, free-flow time) and build a synthetic
   Origin-Destination (O-D) demand matrix.
4. Run a static user-equilibrium traffic assignment (BPR volume-delay function).

Then, four kinds of scenario:

1. **Close a road** (an accident, roadworks, a landslide).
2. **Change capacity or speed** of existing roads (add/remove a lane, change a limit).
3. **Change demand** (a new development, or city-wide growth).
4. **Add a new road** (a bypass or express corridor).

It is written for someone who has **never used AequilibraE** before. Every step is
explained in plain language first, and the modelling logic is wrapped in a handful of
small, reusable functions — `solve_car(...)`, `build_graph(...)`, `compare(...)` — so the
same code can later sit behind a graphical interface: **one button per scenario**.

> **The one idea to hold onto:** a scenario is just *an edit to the inputs, followed by
> re-running the model and comparing the numbers*. Nothing more.


## How AequilibraE thinks about a road model

Before any code, here is the vocabulary. A road model in AequilibraE has five pieces:

| Term | Plain-language meaning |
|---|---|
| **Project** | A folder with a database (`project_database.sqlite`) holding the whole model: roads, zones, results. |
| **Links & Nodes** | **Links** are road segments; **nodes** are the points where they meet (junctions). Each link has a length, a **speed**, a **capacity** (how many vehicles/hour it can carry) and a **free-flow travel time**. |
| **Zones, Centroids, Connectors** | The city is divided into **Traffic Analysis Zones (TAZ)**. Each zone has a **centroid** (a single point standing for "everyone in this zone") joined to the real roads by artificial **connector** links. Trips enter and leave the network through centroids. |
| **O-D matrix** | The **demand**: a table of how many car trips go *from* each zone *to* every other zone. |
| **Graph** | A routing-ready, in-memory copy of the network. The model finds shortest paths on the graph. We can tweak the graph (close a link, change a capacity) *without touching the database on disk*. |
| **Assignment** | The calculation that loads the O-D trips onto the network, choosing routes. We use **user equilibrium**: every driver picks their fastest route until no one can do better by switching — a realistic snapshot of rush-hour congestion. |
| **BPR / VDF** | The **Volume-Delay Function**. As a link fills up, it slows down. BPR is the standard formula: travel time rises steeply once **volume approaches capacity**. |
| **V/C ratio** | **Volume ÷ Capacity** on a link. Below ~0.8 it flows freely; above 1.0 it is over capacity (gridlock). Our congestion maps colour links by V/C. |
| **System travel time** | The single headline number: total **vehicle-minutes** spent by everyone in the network. Lower is better. We compare scenarios by how much they move this number. |

That is the whole mental model. The rest of the notebook is: **build it once**, then
**edit-and-re-run** for each scenario.


## What kind of model is this? (coming from microsimulation)

If your instinct for traffic is a **microsimulation**, set it aside for a moment — a
**static assignment** works very differently, and the difference is the thing to
internalise before reading the rest.

**A microsimulation has a clock.** Individual vehicles enter the network, follow
car-following and lane-changing rules, queue at signals, and you watch congestion build up
and clear second by second. Time moves forward.

**A static assignment has no clock.** There are no individual vehicles and no time-stepping.
It solves for a single **steady-state equilibrium** that represents *one* period. You hand
it a matrix of trip **volumes** for that period, and it finds the flow pattern in which no
driver could reach their destination faster by switching route (**user equilibrium**).

What that means in practice:

- **Only volumes are assigned.** The matrix says "this many trips travel from zone A to
  zone B during the modelled period." The model distributes those volumes across routes; it
  never moves a car through time.
- **Congestion is an average, not an event.** A link does not form a queue that grows and
  drains. Instead its volume/capacity ratio rises, and the volume-delay function (BPR) turns
  that into a higher *average* travel time. There is no spillback, no signal timing, no
  vehicle-to-vehicle interaction — just a smooth "more volume → slower link" curve.
- **One run = one period.** Everything in the matrix is treated as happening at once, in
  steady state. To study a different period (say the afternoon), you run the model again
  with that period's matrix.

So every output in this notebook — link volumes, V/C ratios, total vehicle-minutes — is a
**steady-state average for a single period**, not a time-varying picture of jams forming
and dissolving.

**A consequence to watch — keep the units consistent.** The solver only ever looks at the
*ratio* volume / capacity, so both must describe the same time window. Link **capacity is
stored as vehicles per hour** — a fixed property of the road. So the **demand matrix must
also be on a per-hour basis**: the trips expected in a single (usually peak) hour, *not* a
daily total. If your demand is a daily figure, scale it down to the modelled hour first —
feeding a whole day's trips against an hourly capacity makes every road look gridlocked
(V/C of 5-6). Note the asymmetry: you rescale the **demand** to match the hour; you never
rescale capacity, because it belongs to the network, not to the time window.


## Roadmap

- **Part A — Build the baseline model** (project, road network, zones, attributes, demand). Runs once.
- **Part B — The solver and the metrics** (the reusable functions + the baseline result, cached).
- **Part C — The four scenarios** (each is a small function you can call and compare).

> **Heads-up:** Part A downloads Quito's road network from OpenStreetMap. That step needs
> an internet connection and takes a minute or two. Everything after it is fast.


# Part A — Build the baseline model

## A0. QGIS safeguard (run this first, always)

You most likely **don't** have QGIS installed — in that case this cell changes nothing, so
just run it and move on.

It only matters *if* QGIS **is** present on the machine: AequilibraE would then detect it
and try to load QGIS' graphics bindings, which crashes a plain Python session. To cover
that case, the block below registers a tiny fake module *before* AequilibraE is imported,
forcing the lightweight standalone code path either way. It is harmless when QGIS is
absent, so we always run it — and **it must be the very first thing that runs.**


In [ ]:
import sys
import types

# Force AequilibraE onto its standalone (non-QGIS) code path.
mock_qgis_utils = types.ModuleType("aequilibrae.utils.qgis_utils")
mock_qgis_utils.inside_qgis = False
mock_qgis_utils.rtree_avail = True  # the standalone `rtree` package is installed
sys.modules["aequilibrae.utils.qgis_utils"] = mock_qgis_utils

import os
import shutil
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, Point, LineString

from aequilibrae import Project
from aequilibrae.parameters import Parameters
from aequilibrae.matrix import AequilibraeMatrix
from aequilibrae.paths import TrafficAssignment, TrafficClass

print("AequilibraE imported successfully.")

## A1. Create a fresh project

We resolve the repo's `data/` and `outputs/` folders (works whether the kernel starts in
the repo root or in `notebooks/`), then create a brand-new, empty AequilibraE project.


In [ ]:
_cwd = os.getcwd()
BASE_DIR = _cwd if os.path.isdir(os.path.join(_cwd, "outputs")) else os.path.dirname(_cwd)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Dedicated folder for the interactive HTML maps this notebook writes. Cleared on each run
# so it only ever holds the current set of maps (no stale files from previous runs).
MAPS_DIR = os.path.join(OUTPUTS_DIR, "maps")
if os.path.isdir(MAPS_DIR):
    shutil.rmtree(MAPS_DIR)
os.makedirs(MAPS_DIR, exist_ok=True)

project_path = os.path.join(DATA_DIR, "quito_scenarios_project")
if os.path.exists(project_path):
    shutil.rmtree(project_path)

project = Project()
project.new(project_path)
db_path = os.path.join(project_path, "project_database.sqlite")
print(f"Created a fresh AequilibraE project at: {project_path}")

## A2. Download the road network from OpenStreetMap

`create_from_osm` pulls the real streets for a model area — given as a shapely `Polygon`
in `(longitude, latitude)` order — and turns them into links and nodes. The bounding box
below covers Quito's long north-south valley. **This is the slow step (~1-2 min).**

**What counts as a road for cars.** AequilibraE's default OSM class list for the car mode is
generous: it includes `bridleway` (horse path), `pedestrian` (pedestrian street), `construction`
(closed road) and `escape` (runaway-truck ramp). Downloaded, those become ordinary links with
default speed and capacity, and zone connectors attach to them like any other street — and a
centroid attached to a bridleway loses its trips entirely, because the stub it hangs off is
pruned out of the routing graph as a dead end. The cell below removes them from the car list *before* the query, so they never arrive.
Footways, paths, tracks and cycleways were already excluded by AequilibraE.

> **If the download fails with `Server returned no JSON data ... 406 Not Acceptable`:** that is
> the Overpass server refusing AequilibraE's default *plain-HTTP* endpoint. The cell below
> switches the project's `overpass_endpoint` parameter to HTTPS first, which fixes it. Nothing
> is wrong with the model — the same query over HTTPS is answered normally.


In [ ]:
# [min_lat, min_lon, max_lat, max_lon] spanning Quito's north-south valley.
quito_bbox = [-0.30, -78.55, -0.11, -78.45]
min_lat, min_lon, max_lat, max_lon = quito_bbox

model_area = Polygon([
    (min_lon, min_lat), (max_lon, min_lat),
    (max_lon, max_lat), (min_lon, max_lat), (min_lon, min_lat),
])

# Two things to fix in the project's parameters before downloading.
par = Parameters()
osm, net_osm = par.parameters["osm"], par.parameters["network"]["osm"]

# 1. Overpass endpoint: AequilibraE's default is plain HTTP and the server now rejects those
#    requests with "406 Not Acceptable". The same query over HTTPS is answered normally.
if osm["overpass_endpoint"].startswith("http://"):
    osm["overpass_endpoint"] = "https://overpass-api.de/api"
    print(f"Overpass endpoint set to {osm['overpass_endpoint']}")

# 2. What counts as a road for cars. AequilibraE's default list of OSM classes for the car mode
#    includes `bridleway` (a horse path), `pedestrian` (a pedestrian street), `construction` (a
#    road that is closed) and `escape` (a runaway-truck ramp). Downloaded, they become ordinary
#    links that get default speed and capacity, and zone connectors attach to them like any
#    other street, and a centroid attached to a bridleway loses its trips when that stub is
#    pruned from the graph as a dead end.
#    Dropping them from the car list keeps them out of the OSM query altogether, which is
#    cheaper and safer than deleting links afterwards.
NOT_FOR_CARS = {"bridleway", "pedestrian", "construction", "escape"}
car_types = net_osm["modes"]["car"]["link_types"]
dropped = sorted(set(car_types) & NOT_FOR_CARS)
if dropped:
    net_osm["modes"]["car"]["link_types"] = [t for t in car_types if t not in NOT_FOR_CARS]
    print("Not roads for cars, excluded from the download:", ", ".join(dropped))
par.write_back()

print("Downloading Quito's road network from OSM (this can take a minute or two)...")
project.network.create_from_osm(model_area=model_area, modes=["car"])
print(f"Network loaded. Links: {project.network.count_links()}, "
      f"Nodes: {project.network.count_nodes()}")

# Backstop. AequilibraE excludes only the OSM values listed in its own parameters, and OSM has a
# long tail beyond that list - this bbox delivers `ladder` and `rest_area`, which no filter above
# would have caught. So we finish with a whitelist: keep the classes we price in A4, drop the
# rest. The rule is deliberate - if we cannot state a speed and a capacity for it, it is not a
# road this model should route cars over.
CAR_CLASSES = ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link",
               "secondary", "secondary_link", "tertiary", "tertiary_link",
               "unclassified", "residential", "living_street", "service", "road"]
with project.db_connection as conn:
    counts = pd.read_sql("SELECT link_type, COUNT(*) AS n FROM links GROUP BY link_type", conn)
    extra = sorted(set(counts["link_type"].dropna()) - set(CAR_CLASSES))
    if extra:
        removed = int(counts.loc[counts["link_type"].isin(extra), "n"].sum())
        conn.execute("DELETE FROM links WHERE link_type IN "
                     f"({','.join('?' * len(extra))})", extra)
        conn.commit()
        print(f"Removed {removed} link(s) of non-car classes: {', '.join(extra)}")
project.network.links.refresh_fields()
print(f"Car network: {project.network.count_links()} links.")

## A3. Define the Traffic Analysis Zones (TAZ)

We lay a synthetic **grid of zones** over the area — more rows than columns, because Quito
is far longer than it is wide. The count is set by `N_ROWS`/`N_COLS` below; **more zones
spread demand onto more streets**, so the network coverage is directly controlled here. For
each zone we create its polygon, drop a **centroid** at its centre, and — in one bulk step —
build the **connectors** that join every centroid to the car network. In a real study these
zones would come from official data; here a grid keeps the demo self-contained.


In [ ]:
N_ROWS, N_COLS = 12, 6            # more zones than the original 8x4 -> more streets get used
zoning = project.zoning
lat_edges = np.linspace(min_lat, max_lat, N_ROWS + 1)
lon_edges = np.linspace(min_lon, max_lon, N_COLS + 1)

zone_id = 1
for i in range(N_ROWS):
    for j in range(N_COLS):
        poly = Polygon([
            (lon_edges[j],   lat_edges[i]),   (lon_edges[j+1], lat_edges[i]),
            (lon_edges[j+1], lat_edges[i+1]), (lon_edges[j],   lat_edges[i+1]),
            (lon_edges[j],   lat_edges[i]),
        ])
        zone = zoning.new(zone_id)
        zone.geometry = poly
        zone.add_centroid(Point(poly.centroid.x, poly.centroid.y))
        zone.save()
        zone_id += 1

# Build every centroid connector to the car network ("c") in one bulk operation.
zoning.connect_mode(mode_id="c", bulk=True)
print(f"Created {N_ROWS * N_COLS} zones and {project.network.count_centroids()} centroids.")

## A4. Fill in speed, capacity and free-flow time

An OSM import gives geometry and road classes but leaves **speed** and **capacity**
mostly empty, and the new connectors have no attributes at all. The volume-delay function
needs all three. We assign a speed and capacity to every link from its road class, give
the artificial connectors a huge capacity (so they never become the bottleneck), then
compute **free-flow travel time = distance / speed**.

> These are illustrative defaults. Calibrating them against Quito's real bottlenecks
> (the Túnel de Guayasamín, the Autopista Rumiñahui) is where real modelling work lies.

`SPEED_KMH` and `CAPACITY` cover the ramps (`*_link`) as well as the through classes — a
motorway ramp filed as a 30 km/h side street would distort every path that uses the motorway.
Anything *not* in those tables would silently fall back to `DEFAULT_SPEED` / `DEFAULT_CAP`, so
the cell **names** any class that had to, and prints the classes actually present. If a class
appears that cars should not use, exclude it in A2 rather than pricing it here.


In [ ]:
# Free-flow speed (km/h) and capacity (veh/h) per OSM road class. The `_link` entries are the
# ramps and slip roads that join the big roads: they are numerous, and leaving them out of this
# table would quietly file motorway ramps as 30 km/h side streets.
SPEED_KMH = {"motorway": 90, "motorway_link": 60,
             "trunk": 80, "trunk_link": 55,
             "primary": 60, "primary_link": 45,
             "secondary": 50, "secondary_link": 40,
             "tertiary": 40, "tertiary_link": 35,
             "unclassified": 30, "residential": 30, "living_street": 20,
             "service": 20, "road": 30}
CAPACITY  = {"motorway": 2000, "motorway_link": 1500,
             "trunk": 1900, "trunk_link": 1400,
             "primary": 1800, "primary_link": 1200,
             "secondary": 1400, "secondary_link": 1000,
             "tertiary": 1000, "tertiary_link": 800,
             "unclassified": 800, "residential": 600, "living_street": 400,
             "service": 300, "road": 800}
DEFAULT_SPEED, DEFAULT_CAP = 30, 800

with project.db_connection as conn:
    cur = conn.cursor()
    link_types = [r[0] for r in cur.execute(
        "SELECT DISTINCT link_type FROM links WHERE link_type IS NOT NULL").fetchall()]
    for lt in link_types:
        cur.execute("UPDATE links SET speed_ab=?, speed_ba=?, capacity_ab=?, capacity_ba=? "
                    "WHERE link_type=?",
                    (SPEED_KMH.get(lt, DEFAULT_SPEED), SPEED_KMH.get(lt, DEFAULT_SPEED),
                     CAPACITY.get(lt, DEFAULT_CAP), CAPACITY.get(lt, DEFAULT_CAP), lt))
    # Connectors (and anything still empty): fast + effectively unlimited.
    cur.execute("UPDATE links SET "
                "speed_ab=COALESCE(speed_ab, 40), speed_ba=COALESCE(speed_ba, 40), "
                "capacity_ab=COALESCE(capacity_ab, 100000), capacity_ba=COALESCE(capacity_ba, 100000)")
    # Free-flow travel time in MINUTES.
    cur.execute("UPDATE links SET "
                "travel_time_ab=(distance/1000.0)/speed_ab*60.0, "
                "travel_time_ba=(distance/1000.0)/speed_ba*60.0")
    conn.commit()
# Any class we did not think of would silently take DEFAULT_SPEED / DEFAULT_CAP - that is how a
# horse path ends up modelled as an 800 veh/h street. Say so instead of hiding it.
# (centroid_connector is deliberately absent from the tables - the COALESCE above gives
# connectors their own fast, effectively unlimited values.)
unpriced = sorted(t for t in link_types
                  if t and t not in SPEED_KMH and t != "centroid_connector")
if unpriced:
    print(f"WARNING: no speed/capacity defined for {unpriced} - they fall back to "
          f"{DEFAULT_SPEED} km/h / {DEFAULT_CAP} veh/h. Add them to SPEED_KMH/CAPACITY, or "
          f"exclude them from the download if cars should not use them.")

print("Link attributes populated (speed, capacity, free-flow travel time).")
print("Road classes in the network:", ", ".join(sorted(t for t in link_types if t)))

## A5. The demand (O-D matrix)

We build a reproducible **synthetic** matrix of trip **volumes**: how many car trips
travel between each pair of zones **during the period this model represents**. In a real
study these volumes would come from a survey or a gravity model, already expressed for the
modelled period.

Because the model has no clock (see *What kind of model is this?* above), this one matrix
*is* the demand — there is nothing to spread over time. The volumes are stated **per hour**,
matching the veh/hour capacities from A4, so the volume/capacity ratios are meaningful. Every
scenario re-uses this matrix, and the *demand* scenario scales it.

`baseline_demand` holds those volumes (no trips within a zone).


In [ ]:
num_zones = N_ROWS * N_COLS
centroids = np.arange(1, num_zones + 1)      # centroid ids == zone ids

# Trip volumes for the modelled period, one value per origin-destination pair. We target a
# total number of peak-hour trips and spread it across all O-D pairs, so the loading stays
# sensible whatever the zone count (more zones -> smaller per-pair volumes, not more total
# traffic). Tune TARGET_PEAK_TRIPS if the busiest corridors come out too empty or gridlocked.
TARGET_PEAK_TRIPS = 25_000
mean_per_pair = TARGET_PEAK_TRIPS / (num_zones * (num_zones - 1))

np.random.seed(42)
baseline_demand = np.random.poisson(mean_per_pair, size=(num_zones, num_zones)).astype(float)
np.fill_diagonal(baseline_demand, 0)          # no intra-zonal trips

print(f"{num_zones} zones; total trips in the modelled period: {int(baseline_demand.sum()):,}")

# Part B — The solver and the metrics

Everything the model does reduces to one function:

```
solve_car(network, demand)  ->  metrics
```

Below we build that function once, plus the small helpers around it. A graphical
interface would call exactly these — the buttons just change the *inputs*.


## B1. The building-block functions

Three helpers do all the work:

- **`build_graph(exclude=..., edit=...)`** — makes a fresh routing graph from the current
  network. `exclude` drops links (a *closure*); `edit` is a function that tweaks the
  in-memory edge table (a *capacity/speed change*). Crucially, both act **only in memory** —
  the database on disk is untouched, so the baseline is never corrupted.
- **`make_demand(array)`** — wraps a NumPy O-D array in the matrix object the engine reads.
- **`solve_car(graph, demand, label)`** — runs the user-equilibrium assignment (BPR +
  bi-conjugate Frank-Wolfe) and returns a small `ScenarioResult` holding the per-link
  results and the headline system travel time.

> **Why `build_graph` re-reads the network every time:** each call rebuilds the graph from
> the database, so any in-memory tweak from a previous scenario is automatically forgotten.
> Every scenario therefore starts from the same clean baseline.


In [ ]:
class ScenarioResult:
    """Holds one assignment's output and derives the headline metric."""
    def __init__(self, label, results_df):
        self.label = label
        self.results = results_df                 # indexed by link_id
    @property
    def system_time(self):
        # Total vehicle-minutes across the whole network (the headline KPI).
        r = self.results
        return float((r["car_ab"] * r["Congested_Time_AB"]
                      + r["car_ba"] * r["Congested_Time_BA"]).sum())


def build_graph(exclude=None, edit=None):
    """Fresh car routing graph. `exclude`: link_ids to close. `edit(df)`: tweak edges in place."""
    project.network.build_graphs(modes=["c"])
    g = project.network.graphs["c"]
    g.prepare_graph(centroids)
    if exclude is not None and len(exclude) > 0:
        g.exclude_links(list(exclude))           # remove links from routing (in memory)
    if edit is not None:
        edit(g.graph)                            # tweak capacity/travel_time BEFORE costs are read
    g.set_graph("travel_time")                   # cost for shortest paths
    g.set_skimming(["travel_time"])
    g.set_blocked_centroid_flows(True)           # no shortcuts through zone connectors
    return g


def make_demand(array):
    """Wrap a (num_zones x num_zones) NumPy array as an AequilibraE demand matrix."""
    m = AequilibraeMatrix()
    m.create_empty(zones=num_zones, matrix_names=["car"], memory_only=True)
    m.index[:] = centroids
    m.matrices[:, :, 0] = array
    m.computational_view(["car"])
    return m


def solve_car(graph, demand, label="scenario"):
    """Run one user-equilibrium assignment and return its ScenarioResult."""
    assig = TrafficAssignment()
    assig.add_class(TrafficClass(name="car", graph=graph, matrix=demand))
    assig.set_vdf("BPR")
    assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})
    assig.set_capacity_field("capacity")         # read from graph.graph, after any edit()
    assig.set_time_field("travel_time")          # read from graph.graph, after any edit()
    assig.set_algorithm("bfw")
    assig.max_iter = 100
    assig.rgap_target = 0.001
    assig.execute()
    return ScenarioResult(label, assig.results())

print("Solver helpers defined: build_graph, make_demand, solve_car.")

## B2. A peek at the graph (so it is not a black box)

The graph is just a table of directed edges. Each row is one direction of one link, with
the columns the model reads: `travel_time`, `capacity`, `distance`. When a scenario
"changes capacity", all it does is multiply the `capacity` column for a few rows. Let's
look at it.


In [ ]:
_peek = build_graph()
print("Columns the graph exposes:")
print(list(_peek.graph.columns))
print("\nA few edges (link_id, direction, travel_time [min], capacity [veh/h], distance [m]):")
print(_peek.graph[["link_id", "direction", "travel_time", "capacity", "distance"]].head())

## B3. Make sure every zone can reach the network

`connect_mode` attached each centroid to the nearest node **inside its own zone**. That is a
reasonable default, but it fails in two visible ways:

- a zone with no streets inside it (here: the western slopes) gets **no connector at all**;
- a zone whose nearest node is a **cul-de-sac tip** gets a connector that
  `prepare_graph(remove_dead_ends=True)` promptly throws away with the rest of the stub.

Either way the zone's trips stay in the matrix and never reach the network — the assignment
simply drops them, which is what the `NOT CONNECTED` zones on map 0 and map 1 are telling you.

`fix_zone_connectors()` repairs the repairable. A connector stands for the local streets a zone's
traffic uses to reach the network, so it must be **short and inside the zone**; the search is
ordered that way:

1. the nearest **junction inside the zone** (three or more live streets meeting — pruning keeps
   it, and traffic can leave in more than one direction);
2. failing that, the nearest other node inside the zone that is not a dead-end tip;
3. only if the zone holds no usable street at all, the nearest one within `max_km` (0.75 km) —
   enough to bridge a grid boundary, not enough to invent access.

**A zone with nothing within `max_km` is left unconnected on purpose.** Stretching a connector
over kilometres would inject that zone's trips into the network somewhere they never travel,
quietly loading streets that should be empty — a worse error than the honest gap, and a much
harder one to notice. A TAZ over roadless mountainside genuinely has no road access, so those
zones stay flagged `NOT CONNECTED`, which is the honest outcome.

It does leave the books uneven, so the cell says so: **a trip needs both of its zones on the
network**, therefore one disconnected zone removes a whole row *and* a whole column from the
matrix, and the trips actually loaded are fewer than `TARGET_PEAK_TRIPS`. `assignable_share()`
reports by how much. If you would rather have the two numbers agree, zero those zones' rows and
columns and rescale the matrix — but that is a demand decision, not a network one, so it is not
done here.

This runs **before** the baseline, so every scenario is solved on the same, repaired network.


In [ ]:
import geopandas as gpd
import shapely.wkb
from aequilibrae.utils.spatialite_utils import connect_spatialite


def read_geo(table, columns):
    """Read a project layer WITH geometry, without needing a GDAL engine.

    Uses AequilibraE's SpatiaLite connection to fetch geometry as WKB, then builds a
    GeoDataFrame in memory (geopandas only needs GDAL for read_file, not for construction).
    """
    conn = connect_spatialite(db_path)
    try:
        df = pd.read_sql(f"SELECT {columns}, ST_AsBinary(geometry) AS _wkb FROM {table}", conn)
    finally:
        conn.close()
    geom = [shapely.wkb.loads(bytes(b)) if b is not None else None for b in df["_wkb"]]
    return gpd.GeoDataFrame(df.drop(columns="_wkb"), geometry=geom, crs="EPSG:4326")


def street_links():
    """The links table without the connectors: the actual streets, straight from the database."""
    with project.db_connection as conn:
        df = pd.read_sql("SELECT link_id, link_type, a_node, b_node FROM links", conn)
    return df[df["link_type"] != "centroid_connector"]


def zone_connector_links(centroid_ids=None):
    """``{centroid id: [connector link ids]}`` from the links table.

    Empty when the project has no connectors at all - they have not been built yet.
    """
    with project.db_connection as conn:
        links = pd.read_sql("SELECT link_id, a_node, b_node FROM links "
                            "WHERE link_type = 'centroid_connector'", conn)
    if not len(links):
        return {}
    if centroid_ids is None:
        with project.db_connection as conn:
            centroid_ids = pd.read_sql("SELECT node_id FROM nodes WHERE is_centroid = 1",
                                       conn)["node_id"]
    ids = {int(i) for i in centroid_ids}
    out = {}
    for r in links.itertuples():
        for end in (int(r.a_node), int(r.b_node)):
            if end in ids:                        # only one end of a connector is a centroid
                out.setdefault(end, []).append(int(r.link_id))
    return out


def zone_connectors(centroid_ids=None):
    """``{centroid id: number of centroid connectors}`` - a demand-free existence check.

    This is the test behind AequilibraE's first "centroids not present in the graph" warning:
    it only asks whether a connector exists. `zone_graph_status()` is stronger (it also knows
    whether the connector survived dead-end pruning) but needs a prepared graph.
    """
    return {z: len(ids) for z, ids in zone_connector_links(centroid_ids).items()}


def zone_graph_status(build_if_needed=False):
    """Which centroids the routing graph can actually serve - **from the graph, no demand**.

    Whether a zone can be served is decided by `prepare_graph()`, which reads the network and the
    centroids and nothing else; the demand matrix never enters into it. Two things can go wrong:

    * the centroid has **no connector** in the links table - nothing was ever attached to it;
    * its connectors were **pruned**: `prepare_graph(remove_dead_ends=True)` burns links that lead
      nowhere, and a connector hanging off a cul-de-sac goes with them. The burnt link ids are
      left in `graph.dead_end_links`, so we look for centroids whose every connector is in there.

    Do NOT be tempted to test membership of `graph.all_nodes` / `compact_all_nodes`: those arrays
    are built as ``hstack((centroids, other_nodes))``, so every centroid is present by
    construction, connected or not, and the test would silently always pass.

    Returns ``{zone_id: "ok" | "pruned" | "no connector"}``, or ``{}`` when no graph has been
    prepared and `build_if_needed` is False. Reflects the most recently prepared graph.
    """
    g = (getattr(project.network, "graphs", {}) or {}).get("c")
    if g is None or getattr(g, "dead_end_links", None) is None:
        if not build_if_needed:
            return {}
        g = build_graph()
    dead = {int(x) for x in g.dead_end_links}
    cents = [int(c) for c in g.centroids]
    per_zone = zone_connector_links(cents)
    out = {}
    for c in cents:
        ids = per_zone.get(c, [])
        out[c] = ("no connector" if not ids
                  else "pruned" if all(i in dead for i in ids) else "ok")
    return out


CONNECTOR_SPEED, CONNECTOR_CAPACITY = 40, 100_000   # fast and effectively unlimited, as in A4


def attachable_nodes(min_degree=2, dead=frozenset()):
    """Street nodes a connector may attach to, with the number of live streets meeting there.

    Dead-end **tips** (degree 1) are excluded: `prepare_graph` burns the stub they belong to and
    takes the connector with it. A mid-block node (degree 2) is fine - it is on a through street,
    so it survives and leads somewhere - but a junction (degree 3+) is better, and the search
    below prefers one. Links already burnt as dead ends do not count towards the degree, so a
    junction *inside* a pruned stub does not qualify.
    """
    net = street_links()
    net = net[~net["link_id"].isin(list(dead))]
    degree = pd.concat([net["a_node"], net["b_node"]]).value_counts()
    keep = degree[degree >= min_degree]
    nodes = read_geo("nodes", "node_id, is_centroid")
    nodes = nodes[(nodes["is_centroid"] == 0) & (nodes["node_id"].isin(keep.index))].copy()
    nodes["degree"] = nodes["node_id"].map(keep.to_dict()).astype(int)
    return nodes.reset_index(drop=True)


def _km_from(point, xs, ys):
    """Distances in km from one lon/lat point to arrays of lon/lat (fine at this latitude)."""
    return np.hypot((xs - point.x) * 111.32 * float(np.cos(np.radians(point.y))),
                    (ys - point.y) * 110.57)


def fix_zone_connectors(min_degree=3, max_km=0.75):
    """Re-attach every centroid the graph cannot serve - preferring a junction **inside** its zone.

    A connector stands for the local streets a zone's traffic uses to reach the network, so it has
    to be short and it belongs inside the zone. The search is ordered accordingly:

    1. the nearest **junction inside the zone** (at least `min_degree` live streets meeting);
    2. failing that, the nearest other attachable node inside the zone (a mid-block node is fine);
    3. only if the zone contains no usable street at all, the nearest node within `max_km` of the
       centroid. A grid zone whose nearest street sits just over the boundary is worth bridging;
       a connector kilometres long is not - it would inject the zone's trips into the network
       somewhere they never actually travel, quietly loading streets that should be empty.

    A zone with nothing within `max_km` is **left unconnected on purpose** and reported. That is a
    zoning problem rather than a connector problem: a TAZ over roadless mountainside has no
    business generating trips, and the honest fixes are to redraw the zoning or to zero that
    zone's demand - not to fake access with a long connector.

    Returns the list of ``(zone, node, km)`` it re-attached.
    """
    g = build_graph()
    dead = {int(x) for x in g.dead_end_links}
    status = zone_graph_status()
    broken = sorted(z for z, s in status.items() if s != "ok")
    if not broken:
        print("Every zone can reach the car network - no connectors to fix.")
        return []
    print(f"{len(broken)} zone(s) cannot reach the network: "
          f"{', '.join(f'{z} ({status[z]})' for z in broken)}")

    cand = attachable_nodes(min_degree=2, dead=dead)
    if not len(cand):
        raise RuntimeError("No attachable street node found - is the network empty?")
    zones = read_geo("zones", "zone_id")
    cand = gpd.sjoin(cand, zones[["zone_id", "geometry"]], predicate="within", how="left")
    cand = cand.drop_duplicates("node_id").reset_index(drop=True)
    xs, ys = cand.geometry.x.to_numpy(), cand.geometry.y.to_numpy()

    nodes = read_geo("nodes", "node_id, is_centroid")
    centroid_pt = nodes[nodes["is_centroid"] == 1].set_index("node_id")["geometry"]
    existing = zone_connector_links([int(z) for z in broken])

    links_table = project.network.links
    fixed, given_up = [], []
    for z in broken:
        p = centroid_pt.loc[z]
        km = _km_from(p, xs, ys)
        inside = (cand["zone_id"] == z).to_numpy()
        junction = (cand["degree"] >= min_degree).to_numpy()

        for mask, where in ((inside & junction, "junction in the zone"),
                            (inside, "street in the zone"),
                            (km <= max_km, f"nearest street within {max_km:g} km")):
            if mask.any():
                pick = int(np.flatnonzero(mask)[np.argmin(km[mask])])
                break
        else:
            given_up.append(z)
            print(f"  zone {int(z):>3}: no usable street inside the zone, and none within "
                  f"{max_km:g} km - left unconnected (its {baseline_demand[z - 1, :].sum() + baseline_demand[:, z - 1].sum():,.0f} "
                  f"trips stay unassigned)")
            continue

        target, dist = int(cand["node_id"].iloc[pick]), float(km[pick])
        old = existing.get(int(z), [])
        if old:                                       # drop the unusable connector first
            with project.db_connection as conn:
                conn.execute("DELETE FROM links WHERE link_id IN "
                             f"({','.join('?' * len(old))})", [int(i) for i in old])
                conn.commit()
        links_table.refresh_fields()

        link = links_table.new()
        link.geometry = LineString([(p.x, p.y), (float(xs[pick]), float(ys[pick]))])
        link.modes = "c"
        link.link_type = "centroid_connector"
        link.name = f"connector zone {int(z)}"
        link.direction = 0
        link.speed_ab = link.speed_ba = CONNECTOR_SPEED
        link.capacity_ab = link.capacity_ba = CONNECTOR_CAPACITY
        new_id = link.link_id
        link.save()

        with project.db_connection as conn:
            conn.execute("UPDATE links SET travel_time_ab=(distance/1000.0)/speed_ab*60.0, "
                         "travel_time_ba=(distance/1000.0)/speed_ba*60.0 WHERE link_id=?",
                         (new_id,))
            conn.commit()
            chk = pd.read_sql(f"SELECT a_node, b_node FROM links WHERE link_id={new_id}", conn)
        assert {int(chk.a_node[0]), int(chk.b_node[0])} == {int(z), target}, \
            f"zone {z}: connector did not snap to nodes {z} and {target}"

        print(f"  zone {int(z):>3}: -> node {target} ({dist * 1000:,.0f} m, "
              f"{int(cand['degree'].iloc[pick])} streets, {where})")
        fixed.append((int(z), target, dist))

    links_table.refresh_fields()
    still = sorted(z for z, s in zone_graph_status(build_if_needed=True).items() if s != "ok")
    print(f"Re-attached {len(fixed)} zone(s).")
    if still:
        print(f"Still unreachable: {still} - no street near enough, so they stay disconnected "
              f"(the zone maps keep flagging them as NOT CONNECTED).")
        print(f"  {assignable_share(still)}")
    return fixed


def assignable_demand(unreachable=None):
    """(assignable trips, total trips): the matrix minus everything touching a cut-off zone.

    A trip needs BOTH of its zones on the network, so one disconnected zone removes a whole row
    *and* a whole column. Worth stating explicitly: leaving those zones out is a deliberate
    choice, but it means the trips the model actually loads are fewer than `TARGET_PEAK_TRIPS`,
    and nothing else in the output would tell you by how much.
    """
    if unreachable is None:
        unreachable = [z for z, s in zone_graph_status(build_if_needed=True).items() if s != "ok"]
    ok = [z - 1 for z in range(1, num_zones + 1) if z not in set(unreachable)]
    total = float(baseline_demand.sum())
    return float(baseline_demand[np.ix_(ok, ok)].sum()), total


def assignable_share(unreachable=None):
    """One line of accounting: how much of the matrix can actually be assigned."""
    ok, total = assignable_demand(unreachable)
    return (f"Assignable demand: {ok:,.0f} of {total:,.0f} trips ({100 * ok / total:.1f}%) - "
            f"the rest has at least one end in a disconnected zone.")


fix_zone_connectors()


## B4. Solve the baseline — once — and cache it

This is the reference case. Every scenario is compared against it, so we solve it a single
time and keep the result in `baseline`.


In [ ]:
# Link attributes we will reuse for reporting and maps.
with project.db_connection as conn:
    link_attr = pd.read_sql(
        "SELECT link_id, name, link_type, a_node, b_node, distance FROM links", conn)

baseline = solve_car(build_graph(), make_demand(baseline_demand), label="Baseline")
print(f"Baseline solved. System travel time: {baseline.system_time:,.0f} vehicle-minutes.")

# The busiest real streets in the baseline.
_top = (baseline.results.reset_index().merge(link_attr, on="link_id", how="left"))
_top = _top[_top.link_type != "centroid_connector"]
_top["name"] = _top["name"].fillna("(unnamed)")
print("\nTop 5 busiest streets (baseline):")
print(_top.sort_values("car_tot", ascending=False)
          .head(5)[["name", "link_type", "car_tot", "VOC_max"]].to_string(index=False))

## B5. The comparison and map helpers

- **`compare(base, scen)`** prints the change in system travel time and returns a table of
  which links gained or lost traffic (in vehicles and as a % of the baseline volume) — the raw
  material for a difference map.
- **`map_zone_system()`** draws the zones and their centroids from *geometry alone*, flagging any
  zone the network cannot serve. Needs no demand and no results.
- **`map_zones()`** the same picture, enriched with the demand matrix and the baseline result.
- **`map_congestion(scen)`** draws one scenario coloured by V/C (blue = free, red = over capacity).
- **`map_difference(changes, closed, new)`** draws where traffic moved between two scenarios
  (red = more, blue = less), highlighting any closed or newly-built link.
- **`map_capacity(base, scen, changed_links)`** draws the change in *congestion* (V/C) after a
  capacity edit — the right view for a widening, where a volume map would mislead.

**Every map shares one hover panel** (`_tip`): the feature's name, a coloured tag saying what
it is, then the figures that matter for *that* map. Volumes are whole vehicles and ratios have
two decimals — a static assignment is not precise beyond that.

| Map | What the hover panel shows |
|---|---|
| Zone system (`map_zone_system`) | approximate area, number of centroid connectors, centroid node id |
| Zone system (`map_zones`) | trips leaving / arriving in the zone (matrix and assigned), centroid node id |
| Congestion | class, volume, V/C ratio, % of capacity used |
| Difference (closure, demand, new road) | class, baseline vs scenario volume, change in vehicles **and in per cent**, V/C before &rarr; after |
| &nbsp;&nbsp;&nbsp;- a closed link | volume before, and that it now carries none |
| &nbsp;&nbsp;&nbsp;- a new road | **vehicles it carries**, capacity, V/C, free-flow speed, length |
| Capacity change | V/C before / after / change, then volume before / after / change (veh and %) |

Every panel ends with the `link id`, so anything you see on the map can be looked up in the
database or fed back into a scenario function.

> **What the grey network means.** The legend calls it *"minor change"*, and that is literal:
> grey is *below the highlight threshold*, **not** *unchanged*. An equilibrium assignment nudges thousands of links by a vehicle or two, which
> is solver noise rather than a result, so `map_difference` highlights changes above
> `min_change` (25 veh) and `map_capacity` above `min_dvoc` (0.05 V/C). Both print how many
> links stayed grey — and, for volumes, what share of the total change that represents — and
> both state the threshold in the legend. Lower the parameter to see more. Two further
> omissions to know about: the grey skeleton leaves out the smallest road classes
> (`CONTEXT_ROAD_TYPES`) and never includes centroid connectors.

Together these are the entire "results view" — a GUI would reuse them unchanged. Each map is
also written to an HTML file in `outputs/maps/` (the exact path is printed when you run the
cell), so you can always open it in a browser even if it does not render inline.

> **Reading map geometry without GDAL.** The usual `geopandas.read_file(...)` needs a GDAL
> engine (`pyogrio`/`fiona`), which some locked-down Windows machines block. To stay
> portable we instead read the geometry as well-known-binary through AequilibraE's own
> SpatiaLite connection and build the GeoDataFrame in memory — no external GIS engine
> required. That is all the `read_geo` helper (defined in B3 above) does.


In [ ]:
import folium

# `read_geo`, `zone_connectors` and `zone_graph_status` live in the connectivity step above:
# they are network diagnostics, and the connector repair needs them before this point.

# Road geometry, loaded once (re-read later only when we add a new link).
links_geo = read_geo("links", "link_id, name, link_type")

# Road classes drawn as the grey "context" network behind every map. We drop the smallest
# classes (service roads, footpaths, alleys, ...) to keep the map files a reasonable size.
CONTEXT_ROAD_TYPES = ["motorway", "trunk", "primary", "secondary", "tertiary",
                      "unclassified", "residential", "living_street",
                      "motorway_link", "trunk_link", "primary_link", "secondary_link",
                      "tertiary_link"]                     # ramps, so junctions look joined


# ---------------------------------------------------------------------------------------
# The hover panel
#
# Every map uses ONE tooltip design: the feature's name in bold, an optional coloured tag
# saying what it is, then a two-column table of the figures that matter for that map. The
# look lives in a single stylesheet (`TIP_CSS`) rather than in inline styles, so all maps
# stay consistent and the per-link HTML stays small (these files hold thousands of links).
# ---------------------------------------------------------------------------------------
TIP_CSS = """
<style>
/* folium wraps GeoJson tooltips in a table of its own - strip its spacing. */
.foliumtooltip table { margin: 0 !important; }
.foliumtooltip td, .foliumtooltip th { padding: 0; border: none; }
.aeq-tip { font-family: sans-serif; font-size: 12px; color: #222; }
.aeq-tip .hd { font-weight: bold; margin-bottom: 4px; white-space: nowrap; }
.aeq-tip .tag { color: white; border-radius: 3px; padding: 1px 5px; font-size: 10px;
                margin-left: 6px; vertical-align: middle; white-space: nowrap; }
.aeq-tip table { margin: 0 !important; border-collapse: collapse; font-size: 12px; }
.aeq-tip td.k { color: #555; padding-right: 10px; text-align: left; }
.aeq-tip td.v { text-align: right; white-space: nowrap; }
</style>
"""


def _txt(value, default="unnamed"):
    """A displayable string for a value that may be NULL/NaN."""
    return default if value is None or pd.isna(value) else str(value)


def _fmt_voc(value):
    """Format a volume/capacity ratio, or 'n/a' when it is missing."""
    return "n/a" if value is None or pd.isna(value) else f"{float(value):.2f}"


def _tip_html(title, rows, tag=None, tag_color="#555555"):
    """The hover panel as an HTML string. `rows` is a list of (label, value) pairs.

    Numbers should arrive already formatted - volumes as whole vehicles, ratios to two
    decimals. A static assignment is not precise to the third decimal, so showing more would
    only be false precision.
    """
    tag_html = (f'<span class="tag" style="background:{tag_color}">{tag}</span>'
                if tag else "")
    body = "".join(f'<tr><td class="k">{k}</td><td class="v">{v}</td></tr>' for k, v in rows)
    return (f'<div class="aeq-tip"><div class="hd">{title}{tag_html}</div>'
            f'<table>{body}</table></div>')


def _tip(title, rows, tag=None, tag_color="#555555"):
    """The same hover panel as a folium Tooltip (for markers and PolyLines)."""
    return folium.Tooltip(_tip_html(title, rows, tag=tag, tag_color=tag_color), sticky=True)


def _geojson_tip():
    """Hover panel for a GeoJson layer: expects a pre-rendered `tip` column of HTML."""
    return folium.GeoJsonTooltip(fields=["tip"], labels=False, sticky=True)


def _context_layer(m):
    """Draw the drivable road network in light grey behind the data, and fit the map to it."""
    ctx = links_geo[links_geo["link_type"].isin(CONTEXT_ROAD_TYPES)]
    if len(ctx):
        folium.GeoJson(
            ctx[["geometry"]],
            style_function=lambda f: {"color": "#c4c4c4", "weight": 1, "opacity": 1.0},
        ).add_to(m)
        minx, miny, maxx, maxy = ctx.total_bounds
        m.fit_bounds([[miny, minx], [maxy, maxx]])


def _base_map():
    """A blank map carrying the tooltip stylesheet and the grey context network."""
    m = folium.Map(tiles="CartoDB positron")
    m.get_root().header.add_child(folium.Element(TIP_CSS))
    _context_layer(m)
    return m


def _save_map(m, filename):
    """Save a folium map into MAPS_DIR, print where it went, and return it for inline display."""
    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}")
    print(f"  location: {os.path.abspath(path)}")
    return m


def _add_legend(m, title, rows, note=None):
    """Add a small floating legend. `rows` is a list of (colour, label) pairs."""
    items = "".join(
        f'<div><span style="display:inline-block;width:14px;height:3px;'
        f'background:{c};margin:0 6px 3px 0;vertical-align:middle;"></span>{lab}</div>'
        for c, lab in rows)
    note_html = (f'<div style="margin-top:5px;color:#555;font-size:11px;">{note}</div>'
                 if note else "")
    html = (
        '<div style="position:fixed; bottom:24px; left:24px; z-index:9999; background:white; '
        'padding:9px 12px; border:1px solid #999; border-radius:6px; '
        'font-family:sans-serif; font-size:13px; line-height:1.45;">'
        f'<b>{title}</b>{items}{note_html}</div>')
    m.get_root().html.add_child(folium.Element(html))


def _add_info(m, title, lines):
    """Add a small info panel (top-right) with a title and a few lines of text/HTML."""
    body = "".join(f"<div>{ln}</div>" for ln in lines)
    html = (
        '<div style="position:fixed; top:20px; right:20px; z-index:9999; background:white; '
        'padding:9px 12px; border:1px solid #999; border-radius:6px; '
        'font-family:sans-serif; font-size:13px; line-height:1.5;">'
        f'<b>{title}</b>{body}</div>')
    m.get_root().html.add_child(folium.Element(html))


def _add_time_info(m, base_time, scen_time, extra=None):
    """Info panel comparing baseline vs scenario system travel time, with the % change.

    `extra` is an optional list of additional HTML lines shown under the comparison (used to
    report how many vehicles a newly added road carries).
    """
    if base_time is None or scen_time is None:
        return
    pct = 100.0 * (scen_time - base_time) / base_time
    word = "increase" if pct > 0.005 else ("decrease" if pct < -0.005 else "no change")
    color = "#d73027" if pct > 0.005 else ("#1a9850" if pct < -0.005 else "#555555")
    _add_info(m, "System travel time (veh-min)", [
        f"Baseline: {base_time:,.0f}",
        f"Scenario: {scen_time:,.0f}",
        f'Change: <span style="color:{color}; font-weight:bold;">{pct:+.2f}% ({word})</span>',
    ] + list(extra or []))


def compare(base, scen):
    """Print the headline delta and return a per-link change table (real streets only).

    Columns: `vol_base` / `vol_scen` (vehicles), `change` (vehicles), `change_pct` (the change
    as a percentage of the baseline volume) and the V/C ratio in both cases. The join is an
    OUTER one, so a link that exists only in the scenario - a road we just added - is included
    too, with a baseline volume of zero; that is how the new road's volume reaches the map.
    """
    d = scen.system_time - base.system_time
    print(f"{base.label:>16}: {base.system_time:>14,.0f} veh-min")
    print(f"{scen.label:>16}: {scen.system_time:>14,.0f} veh-min   ({100 * d / base.system_time:+.2f}%)")
    b = base.results[["car_tot", "VOC_max"]].rename(
        columns={"car_tot": "vol_base", "VOC_max": "voc_base"})
    s = scen.results[["car_tot", "VOC_max"]].rename(
        columns={"car_tot": "vol_scen", "VOC_max": "voc_scen"})
    c = b.join(s, how="outer")
    c[["vol_base", "vol_scen"]] = c[["vol_base", "vol_scen"]].fillna(0.0)
    c["change"] = c["vol_scen"] - c["vol_base"]
    # Percentage of the baseline volume; undefined (NaN) where the link carried no traffic.
    c["change_pct"] = np.where(c["vol_base"] > 0, 100.0 * c["change"] / c["vol_base"], np.nan)
    c = c.reset_index().merge(link_attr, on="link_id", how="left")
    c = c[c["link_type"] != "centroid_connector"].copy()
    c["name"] = c["name"].fillna("(unnamed)")
    return c


def _voc_color(voc):
    if voc < 0.5:  return "#2c7fb8"   # free-flow
    if voc < 0.8:  return "#41ab5d"
    if voc < 1.0:  return "#feb24c"   # near capacity
    return "#e31a1c"                  # over capacity


def _voc_label(voc):
    if voc < 0.5:  return "free-flowing"
    if voc < 0.8:  return "flowing"
    if voc < 1.0:  return "near capacity"
    return "over capacity"


def map_congestion(scen, filename="congestion.html"):
    """Map one scenario (V/C colour, volume-scaled width); save to disk and return it.

    Hovering a street reports its volume in whole vehicles and its V/C ratio to two decimals -
    which is the same information as "percent of capacity used", so only one of them is shown.
    """
    net = links_geo.merge(scen.results.reset_index()[["link_id", "car_tot", "VOC_max"]],
                          on="link_id", how="inner")
    net = net[(net.link_type != "centroid_connector") & (net.car_tot > 0)].copy()
    net["name"] = net["name"].fillna("unnamed")
    net = net.sort_values("car_tot")               # busiest drawn last (on top)
    max_vol = float(net["car_tot"].max())

    # Hover panel, pre-rendered per link (a GeoJson layer carries it as a `tip` property).
    net["tip"] = [
        _tip_html(r["name"], [
            ("Class", _txt(r["link_type"], "unknown")),
            ("Volume", f"{r['car_tot']:,.0f} veh"),
            ("V/C ratio", f"{r['VOC_max']:.2f}"),
            ("Link id", f"{int(r['link_id'])}"),
        ], tag=_voc_label(r["VOC_max"]), tag_color=_voc_color(r["VOC_max"]))
        for _, r in net.iterrows()]

    m = _base_map()                               # grey full road network + fit to it
    folium.GeoJson(
        net[["tip", "car_tot", "VOC_max", "geometry"]],
        style_function=lambda f: {
            "color": _voc_color(f["properties"]["VOC_max"]),
            "weight": 1.5 + 7 * (f["properties"]["car_tot"] / max_vol),
            "opacity": 0.85},
        tooltip=_geojson_tip(),
    ).add_to(m)
    _add_legend(m, "Congestion (V/C)", [
        ("#2c7fb8", "&lt; 0.5 free-flow"),
        ("#41ab5d", "0.5 &ndash; 0.8"),
        ("#feb24c", "0.8 &ndash; 1.0 near capacity"),
        ("#e31a1c", "&gt; 1.0 over capacity"),
    ], note="line width &prop; volume &middot; hover a street for its numbers")
    _add_info(m, "System travel time (veh-min)", [f"{scen.system_time:,.0f}"])
    return _save_map(m, filename)


def map_difference(changes, closed=None, new=None, mark_zones=None, base_time=None,
                   scen_time=None, filename="difference.html", min_change=25.0,
                   info_extra=None):
    """Map where traffic moved (red=more, blue=less; closed=black, new=green); save and return.

    The whole arterial network is drawn as a faint grey skeleton for context, so the map is
    never near-empty; links with a notable change are highlighted on top.

    Hovering a highlighted link reports its baseline volume, its scenario volume, the change
    both in vehicles and as a percentage of the baseline, and the V/C ratio before and after.
    A newly added road reports the vehicles it carries plus its design attributes.

    `info_extra` adds lines to the top-right panel. Use it to state a scenario's headline input
    once - a demand change, for instance, marks two zones with the *same* number of added trips,
    and only a single statement of the flow tells the reader whether that is one movement or two.

    `min_change` (vehicles) is the highlight threshold, and it is worth being clear about what
    the grey skeleton therefore means: **grey is "not highlighted", not "unchanged"**. A static
    assignment nudges thousands of links by a vehicle or two - noise from the equilibrium, not a
    result - so drawing them all would bury the real story. The function reports how many links
    fall below the threshold and what share of the total change they hold, and the legend states
    the number, so nothing is hidden silently. Lower `min_change` to see more.

    Two other things the grey layer is not: it omits the smallest road classes (see
    `CONTEXT_ROAD_TYPES`) and it never includes centroid connectors.
    """
    closed = set(int(x) for x in (closed or []))
    new = set(int(x) for x in (new or []))
    cols = ["link_id", "vol_base", "vol_scen", "change", "change_pct", "voc_base", "voc_scen"]
    diff = links_geo.merge(changes[cols], on="link_id", how="left")
    diff = diff[diff["link_type"] != "centroid_connector"].copy()
    for col in ["change", "vol_base", "vol_scen"]:
        diff[col] = diff[col].fillna(0.0)
    diff["special"] = diff["link_id"].apply(
        lambda i: "closed" if i in closed else ("new" if i in new else ""))

    # Design attributes of any newly added link, so its tooltip can show what was built.
    specs = {}
    if new:
        ids = ",".join(str(i) for i in sorted(new))
        with project.db_connection as conn:
            specs = (pd.read_sql("SELECT link_id, speed_ab, capacity_ab, distance "
                                 f"FROM links WHERE link_id IN ({ids})", conn)
                     .set_index("link_id").to_dict("index"))

    m = _base_map()                               # grey full road network + fit to it

    # Highlight layer: links whose change clears `min_change`, plus any closed / new link.
    draw = diff[(diff["change"].abs() > min_change) | (diff["special"] != "")].copy()
    ref = draw.loc[draw["special"] == "", "change"].abs()
    max_change = float(ref.max()) if len(ref) and ref.max() > 0 else 1.0
    for _, row in draw.reindex(draw["change"].abs().sort_values().index).iterrows():
        geo = row.geometry
        if geo is None or geo.geom_type != "LineString":
            continue
        nm = _txt(row.get("name"))
        klass = _txt(row.get("link_type"), "unknown")
        link_id = int(row["link_id"])
        vol_base, vol_scen = float(row["vol_base"]), float(row["vol_scen"])
        if row["special"] == "closed":
            color, weight = "#000000", 6
            tip = _tip(nm, [
                ("Class", klass),
                ("Volume before", f"{vol_base:,.0f} veh"),
                ("Volume after", "0 veh (closed)"),
                ("V/C before", _fmt_voc(row["voc_base"])),
                ("Link id", f"{link_id}"),
            ], tag="CLOSED", tag_color="#000000")
        elif row["special"] == "new":
            color, weight = "#12a150", 7
            spec = specs.get(link_id, {})
            rows = [("Class", klass), ("Vehicles carried", f"<b>{vol_scen:,.0f} veh</b>")]
            if spec.get("capacity_ab") is not None:
                rows.append(("Capacity", f"{float(spec['capacity_ab']):,.0f} veh/h"))
            rows.append(("V/C ratio", _fmt_voc(row["voc_scen"])))
            if spec.get("speed_ab") is not None:
                rows.append(("Free-flow speed", f"{float(spec['speed_ab']):,.0f} km/h"))
            if spec.get("distance") is not None:
                rows.append(("Length", f"{float(spec['distance']) / 1000:.2f} km"))
            rows.append(("Link id", f"{link_id}"))
            tip = _tip(nm, rows, tag="NEW ROAD", tag_color="#12a150")
        else:
            color = "#d73027" if row["change"] > 0 else "#4575b4"
            weight = 1.5 + 5 * min(abs(row["change"]) / max_change, 1)
            pct = row["change_pct"]
            # No baseline traffic -> a percentage is undefined, so say so instead.
            pct_txt = f"{pct:+.1f}%" if pd.notna(pct) else "new traffic"
            tip = _tip(nm, [
                ("Class", klass),
                ("Baseline volume", f"{vol_base:,.0f} veh"),
                ("Scenario volume", f"{vol_scen:,.0f} veh"),
                ("Change", f'<b style="color:{color};">{row["change"]:+,.0f} veh</b>'),
                ("Change (%)", f'<b style="color:{color};">{pct_txt}</b>'),
                ("V/C", f'{_fmt_voc(row["voc_base"])} &rarr; {_fmt_voc(row["voc_scen"])}'),
                ("Link id", f"{link_id}"),
            ], tag=("more traffic" if row["change"] > 0 else "less traffic"), tag_color=color)
        pts = [[lat, lon] for lon, lat in geo.coords]
        folium.PolyLine(pts, color=color, weight=weight, opacity=0.9, tooltip=tip).add_to(m)

    # Optional zone markers, e.g. the origin and destination of a demand change. An entry is
    # (zone_id, tag) or (zone_id, tag, rows).
    #
    # When rows are given they REPLACE the zone's baseline trips rather than sitting above them:
    # this is a *change* map, and showing "+1,500 veh added" next to the zone's 300-odd baseline
    # trips invites the reader to compare two numbers that answer different questions. The
    # baseline figures belong on the zone maps, where that is the subject.
    if mark_zones:
        cents = read_geo("nodes", "node_id, is_centroid")
        cents = cents[cents["is_centroid"] == 1]
        for entry in mark_zones:
            zid, tag = entry[0], entry[1]
            rows_extra = list(entry[2]) if len(entry) > 2 else []
            rows_zone = (rows_extra + [("Centroid node id", f"{int(zid)}")] if rows_extra
                         else _zone_rows(int(zid)))
            r = cents[cents["node_id"] == int(zid)]
            if len(r):
                p = r.geometry.iloc[0]
                folium.CircleMarker(
                    [p.y, p.x], radius=8, color="#6a3d9a", fill=True, fill_color="#6a3d9a",
                    fill_opacity=1.0, weight=2,
                    tooltip=_tip(f"Zone {int(zid)}", rows_zone,
                                 tag=tag, tag_color="#6a3d9a")).add_to(m)

    # Be explicit that grey means "below the threshold", not "unchanged".
    below = diff[(diff["change"].abs() <= min_change) & (diff["special"] == "")]
    small = float(below["change"].abs().sum())
    total = float(diff["change"].abs().sum())
    share = 100.0 * small / total if total > 0 else 0.0
    print(f"  {len(below):,} links changed by <= {min_change:g} veh and stay grey "
          f"({share:.1f}% of all volume change); {len(draw):,} links highlighted.")

    # Short label on the swatch; the exact threshold lives in the note below the legend.
    rows = [("#9e9e9e", "minor change")]
    if closed:
        rows.append(("#000000", "closed road"))
    if new:
        rows.append(("#12a150", "new road"))
    rows += [("#d73027", "more traffic"), ("#4575b4", "less traffic")]
    if mark_zones:
        rows.append(("#6a3d9a", "origin / destination zone"))
    _add_legend(m, "Change vs baseline", rows,
                note=f"width &prop; size of change &middot; hover for the numbers<br>"
                     f"grey: moved &le; {min_change:g} veh &mdash; {len(below):,} links, "
                     f"{share:.1f}% of all change")

    # Report the new road's own loading in the info panel, not only in its tooltip.
    extra = []
    for lid in sorted(new):
        r = diff[diff["link_id"] == lid]
        if len(r):
            extra.append(f'<div style="margin-top:5px; border-top:1px solid #ddd; '
                         f'padding-top:5px;">New road carries: '
                         f'<b>{float(r["vol_scen"].iloc[0]):,.0f} veh</b></div>')
    _add_time_info(m, base_time, scen_time, extra=extra + list(info_extra or []))
    return _save_map(m, filename)


def map_capacity(base, scen, changed_links, filename="capacity_change.html",
                 min_dvoc=0.05):
    """Map the CONGESTION change (V/C) after a capacity edit.

    Blue = less congested (improved), red = more congested. The links whose capacity was
    changed get a gold halo so the intervention is obvious. This answers "did it get
    better?" - which a volume-difference map cannot, because a widened road usually carries
    *more* traffic (it just does so with less congestion, which a volume map paints red).

    The hover panel therefore leads with V/C before/after, and shows the volume change (in
    vehicles and per cent) underneath - the two together are the story of the intervention.

    `min_dvoc` is the highlight threshold: links whose V/C moved by less than that stay grey.
    As on the difference map, **grey means "below the threshold", not "unchanged"** - the count
    and the threshold are printed and put in the legend so you can judge what is being left out.
    """
    changed = set(int(x) for x in changed_links)
    b = base.results.reset_index()[["link_id", "car_tot", "VOC_max"]].rename(
        columns={"car_tot": "vol_base", "VOC_max": "voc_base"})
    s = scen.results.reset_index()[["link_id", "car_tot", "VOC_max"]].rename(
        columns={"car_tot": "vol_scen", "VOC_max": "voc_scen"})
    d = b.merge(s, on="link_id", how="inner")
    d["dvoc"] = d["voc_scen"] - d["voc_base"]
    d["dvol"] = d["vol_scen"] - d["vol_base"]
    d["dvol_pct"] = np.where(d["vol_base"] > 0, 100.0 * d["dvol"] / d["vol_base"], np.nan)
    g = links_geo.merge(d, on="link_id", how="inner")
    g = g[g["link_type"] != "centroid_connector"].copy()

    m = _base_map()                               # grey full road network + fit to it

    def dcolor(dv):
        if dv <= -0.20: return "#08519c"   # much less congested (improved)
        if dv <= -0.05: return "#6baed6"   # less congested
        if dv <   0.05: return "#dddddd"   # negligible
        if dv <   0.20: return "#fc9272"   # more congested
        return "#d73027"                   # much more congested

    def dlabel(dv):
        if dv <= -min_dvoc: return ("less congested", "#08519c")
        if dv <   min_dvoc: return ("about the same", "#888888")
        return ("more congested", "#d73027")

    draw = g[(g["dvoc"].abs() > min_dvoc) | (g["link_id"].isin(changed))].copy()
    below = g[(g["dvoc"].abs() <= min_dvoc) & (~g["link_id"].isin(changed))]
    print(f"  {len(below):,} links moved by <= {min_dvoc:g} V/C and stay grey; "
          f"{len(draw):,} links highlighted.")
    mx = float(draw["dvoc"].abs().max()) if len(draw) and draw["dvoc"].abs().max() > 0 else 1.0

    # Gold halo under the modified links, so the intervention location is obvious.
    for _, row in draw[draw["link_id"].isin(changed)].iterrows():
        geo = row.geometry
        if geo is None or geo.geom_type != "LineString":
            continue
        pts = [[lat, lon] for lon, lat in geo.coords]
        folium.PolyLine(pts, color="#ffcc00", weight=10, opacity=0.9).add_to(m)

    # Congestion-change coloured links on top (biggest change drawn last).
    for _, row in draw.reindex(draw["dvoc"].abs().sort_values().index).iterrows():
        geo = row.geometry
        if geo is None or geo.geom_type != "LineString":
            continue
        w = 2 + 5 * min(abs(row["dvoc"]) / mx, 1)
        is_changed = int(row["link_id"]) in changed
        tag, tag_color = (("CAPACITY CHANGED", "#c79100") if is_changed
                          else dlabel(row["dvoc"]))
        pct = row["dvol_pct"]
        pct_txt = f"{pct:+.1f}%" if pd.notna(pct) else "new traffic"
        vcolor = "#d73027" if row["dvol"] > 0 else "#4575b4"
        tip = _tip(_txt(row.get("name")), [
            ("Class", _txt(row.get("link_type"), "unknown")),
            ("V/C before", _fmt_voc(row["voc_base"])),
            ("V/C after", _fmt_voc(row["voc_scen"])),
            ("V/C change", f'<b style="color:{dcolor(row["dvoc"])};">{row["dvoc"]:+.2f}</b>'),
            ("Volume before", f"{row['vol_base']:,.0f} veh"),
            ("Volume after", f"{row['vol_scen']:,.0f} veh"),
            ("Volume change", f'<b style="color:{vcolor};">{row["dvol"]:+,.0f} veh '
                              f'({pct_txt})</b>'),
            ("Link id", f"{int(row['link_id'])}"),
        ], tag=tag, tag_color=tag_color)
        pts = [[lat, lon] for lon, lat in geo.coords]
        folium.PolyLine(pts, color=dcolor(row["dvoc"]), weight=w, opacity=0.95,
                        tooltip=tip).add_to(m)

    _add_legend(m, "Congestion change (V/C)", [
        ("#ffcc00", "capacity changed here"),
        ("#08519c", "much less congested"),
        ("#6baed6", "less congested"),
        ("#fc9272", "more congested"),
        ("#d73027", "much more congested"),
        ("#9e9e9e", "minor change"),
    ], note=f"gold = the modified links &middot; width &prop; size of V/C change &middot; "
            f"hover for the numbers<br>grey: moved &le; {min_dvoc:g} V/C &mdash; "
            f"{len(below):,} links")
    # Report the works as corridors AND links: OSM splits a street at every intersection, so
    # one avenue is dozens of links - "60 links" and "3 roads" are the same intervention.
    names = sorted(set(links_geo.loc[links_geo["link_id"].isin(changed), "name"].dropna()))
    km = float(link_attr.loc[link_attr["link_id"].isin(changed), "distance"].sum()) / 1000.0
    corridors = (f"{len(names)} corridor{'' if len(names) == 1 else 's'}" if names
                 else "selection")
    improved = int((draw["dvoc"] <= -min_dvoc).sum())
    worsened = int((draw["dvoc"] >= min_dvoc).sum())
    _add_time_info(m, base.system_time, scen.system_time, extra=[
        f'<div style="margin-top:5px; border-top:1px solid #ddd; padding-top:5px;">'
        f'Widened: {corridors} &mdash; {len(changed):,} links, {km:.1f} km</div>',
        f'<span style="color:#08519c;">{improved:,} links less congested</span>',
        f'<span style="color:#d73027;">{worsened:,} links more congested</span>'])
    return _save_map(m, filename)


CUT_OFF_COLOR = "#e6550d"          # zones whose trips cannot be assigned

# What the maps say about a zone the network cannot serve. There is more than one underlying
# cause (no connector was ever built; the connector was pruned as a dead end; the connector
# exists but the assignment used none of it) and they matter to whoever *fixes* the model - so
# they are reported in the cell output and returned by zone_graph_status() / zone_connection().
# The map itself says only this, because the reader's conclusion is the same either way:
# these trips are missing from the results, so do not trust this corner of the picture.
CUT_OFF_ROW = ("On the network",
               f'<b style="color:{CUT_OFF_COLOR};">not connected &ndash; its trips '
               f'cannot be assigned</b>')

_ZONE_CONN = None                  # cache for zone_connection()


def zone_connection(refresh=False):
    """Per zone: its centroid connectors, and how many trips actually loaded onto them.

    AequilibraE warns *"Found centroids not present in the graph!"* when a zone's centroid never
    made it into the routing graph - usually because no connector could be built (the centroid
    fell outside the downloaded network) or because its only link was pruned as a dead end.
    Those zones still have trips in the demand matrix, but the assignment silently drops them,
    so a map that just showed their demand would be lying by omission.

    We detect it from the *result* rather than from the graph internals: a zone that is properly
    attached loads its trips onto its centroid connectors, so zero volume there (or no connector
    at all) means the zone is cut off - whatever the reason. The two reasons are reported apart
    because they are different problems:

    * **no centroid connector was built** - the zoning step never attached this centroid. There
      was no network node to attach it to, usually because the centroid sits where the downloaded
      network has nothing (outside the extract, or a hole in it). Visible in the links table, so
      `map_zone_system` can flag it without any assignment.
    * **connector built, but unusable in the graph** - the connector row *exists*, yet the
      assignment put nothing on it: the graph never used it. Typically the node it attaches to is
      a dead-end stub (`prepare_graph` prunes those) or sits on an isolated fragment of network,
      so no route can pass. Only a solved baseline reveals this one.

    Caveat worth knowing: the test reads volume, so a zone with **no demand at all** would also
    show as cut off. Every zone here has trips, so it cannot happen in this notebook - but a
    real model with empty zones would need the demand checked first.

    The split into outbound / inbound is what makes the numbers comparable to the matrix. A
    connector's AB direction runs away from the centroid when `a_node` is the centroid (BA when
    it is `b_node`), and because the graph blocks through-traffic at centroids
    (`set_blocked_centroid_flows(True)`) a connector carries *only* that zone's own trips -
    nothing else is mixed in.

    Returns ``{zone_id: {"connectors": n, "out": veh, "in": veh, "loaded": out + in}}``, or
    ``{}`` before the baseline is solved (the zone map can legitimately be drawn first).
    """
    global _ZONE_CONN
    if _ZONE_CONN is not None and not refresh:
        return _ZONE_CONN
    base = globals().get("baseline")
    if base is None:
        return {}

    conn = (link_attr[link_attr["link_type"] == "centroid_connector"]
            .merge(base.results[["car_ab", "car_ba"]].reset_index(), on="link_id", how="left"))
    conn[["car_ab", "car_ba"]] = conn[["car_ab", "car_ba"]].fillna(0.0)
    at_a = conn["a_node"].isin(centroids)                 # which end of the link is the centroid
    conn["zone_id"] = np.where(at_a, conn["a_node"], conn["b_node"])
    conn["out"] = np.where(at_a, conn["car_ab"], conn["car_ba"])     # leaving the zone
    conn["inb"] = np.where(at_a, conn["car_ba"], conn["car_ab"])     # arriving in the zone
    conn = conn[conn["zone_id"].isin(centroids)]
    agg = conn.groupby("zone_id").agg(connectors=("link_id", "count"),
                                      out=("out", "sum"), inb=("inb", "sum"))

    out = {}
    for z in centroids:
        z = int(z)
        if z in agg.index:
            r = agg.loc[z]
            out[z] = {"connectors": int(r["connectors"]), "out": float(r["out"]),
                      "in": float(r["inb"]), "loaded": float(r["out"] + r["inb"])}
        else:
            out[z] = {"connectors": 0, "out": 0.0, "in": 0.0, "loaded": 0.0}
    _ZONE_CONN = out
    return out


def zone_is_cut_off(info):
    """True when a zone's trips cannot reach the network (no connector, or nothing loaded)."""
    return info["connectors"] == 0 or info["loaded"] <= 0


def _zone_rows(zone_id, conn_info=None):
    """Hover-panel rows for one zone: its demand, and how much of that demand actually travels.

    The **matrix** figures are inputs: the row sum of the demand matrix for trips leaving, the
    column sum for trips arriving. Nothing about the network affects them.

    The **assigned** figures are the loading on the zone's centroid connectors, split the same
    way. They come out slightly *lower* than the matrix, because a trip can only be assigned if
    the zone at its other end is reachable too - the demand of any cut-off zone is lost from
    every partner zone's total.
    """
    i = int(zone_id) - 1                          # zone ids are 1..num_zones, in matrix order
    if not (0 <= i < num_zones):
        return [("Zone id", f"{int(zone_id)}")]
    info = (zone_connection() if conn_info is None else conn_info).get(int(zone_id))
    live = info is not None and not zone_is_cut_off(info)

    rows = [("Trips leaving (matrix)", f"{baseline_demand[i, :].sum():,.0f} veh")]
    if live:
        rows.append(("Trips leaving (assigned)", f"{info['out']:,.0f} veh"))
    rows.append(("Trips arriving (matrix)", f"{baseline_demand[:, i].sum():,.0f} veh"))
    if live:
        rows.append(("Trips arriving (assigned)", f"{info['in']:,.0f} veh"))
    elif info is not None:
        rows.append(CUT_OFF_ROW)
    rows.append(("Centroid node id", f"{int(zone_id)}"))
    return rows


def _zone_area_km2(poly):
    """Rough area in km2 of a lon/lat polygon - fine for city-sized cells at this latitude.

    Scale the bounding box from degrees to km, then keep the shape's share of that box. This
    avoids a re-projection (`to_crs`) so the map needs nothing beyond shapely.
    """
    minx, miny, maxx, maxy = poly.bounds
    box = (maxx - minx) * (maxy - miny)
    if box <= 0:
        return 0.0
    width = (maxx - minx) * 111.32 * float(np.cos(np.radians((miny + maxy) / 2)))
    height = (maxy - miny) * 110.57
    return float(width * height * (poly.area / box))


def map_zone_system(filename="0_zones_and_centroids.html", roads=True):
    """Zones and their centroids, and nothing else: **no demand, no assignment, no results**.

    This is the map you can draw the moment the zoning exists - it answers only "where are my
    zones, where do their centroids sit, and is each one attached to the network?".
    `map_zones` is the richer version of the same picture, but it reads the demand matrix and
    the baseline result to report trips; this one touches geometry and the links table alone,
    so it also works in a project that has no matrix yet (a GUI would show it right after the
    user draws or imports a zoning).

    Zones with **no centroid connector** are drawn in orange and tagged ``NOT CONNECTED`` -
    the same warning as on `map_zones`, reached without needing an assignment. When the project
    has no connectors at all they have simply not been built, and nothing is flagged.

    Set `roads=False` to drop the grey network too - useful before a network is imported.
    """
    zones = read_geo("zones", "zone_id")
    cents = read_geo("nodes", "node_id, is_centroid")
    cents = cents[cents["is_centroid"] == 1]

    # Two demand-free checks, best first: the prepared graph knows whether a connector was
    # built AND whether it survived dead-end pruning; the links table alone only knows the first.
    counts = zone_connectors(cents["node_id"])
    status = zone_graph_status()
    # Causes, for the cell output only - the map shows a single "not connected" category.
    CUT = {"no connector": "no centroid connector was built",
           "pruned": "connector pruned from the graph (hangs off a dead end)"}

    def state(zone_id):
        if status:
            return status.get(int(zone_id), "ok")
        if counts:
            return "ok" if counts.get(int(zone_id), 0) else "no connector"
        return "unknown"                          # connectors not built yet

    cut_off = {int(z) for z in cents["node_id"] if state(z) in CUT}
    checked = bool(status) or bool(counts)

    def rows(zone_id, area=None):
        out = []
        if area is not None:
            out.append(("Area (approx.)", f"{area:,.1f} km&sup2;"))
        st = state(zone_id)
        if st in CUT:
            out.append(CUT_OFF_ROW)
        elif checked:
            out.append(("Centroid connectors", f"{counts.get(int(zone_id), 0)}"))
        out.append(("Centroid node id", f"{int(zone_id)}"))
        return out

    def tag(zone_id, normal, color):
        off = int(zone_id) in cut_off
        return ("NOT CONNECTED" if off else normal, CUT_OFF_COLOR if off else color)

    zones["area_km2"] = [_zone_area_km2(g) for g in zones.geometry]
    zones["cut_off"] = zones["zone_id"].astype(int).isin(cut_off)
    zones["tip"] = [_tip_html(f"Zone {int(r.zone_id)}", rows(r.zone_id, r.area_km2),
                              *tag(r.zone_id, "TAZ", "#3182bd"))
                    for r in zones.itertuples()]

    if roads:
        m = _base_map()                           # grey road network for orientation
    else:
        m = folium.Map(tiles="CartoDB positron")
        m.get_root().header.add_child(folium.Element(TIP_CSS))
    folium.GeoJson(
        zones[["tip", "cut_off", "geometry"]],
        style_function=lambda f: (
            {"color": CUT_OFF_COLOR, "weight": 2, "dashArray": "5,4", "fill": True,
             "fillColor": CUT_OFF_COLOR, "fillOpacity": 0.18}
            if f["properties"]["cut_off"] else
            {"color": "#3182bd", "weight": 1.5, "fill": True,
             "fillColor": "#3182bd", "fillOpacity": 0.06}),
        tooltip=_geojson_tip(),
    ).add_to(m)
    for _, c in cents.iterrows():
        zid, p = int(c.node_id), c.geometry
        off = zid in cut_off
        folium.CircleMarker(
            [p.y, p.x], radius=6 if off else 4,
            color=CUT_OFF_COLOR if off else "#de2d26", fill=True,
            fill_color="#ffffff" if off else "#de2d26",     # hollow = attached to nothing
            fill_opacity=1.0, weight=3 if off else 1,
            tooltip=_tip(f"Zone {zid}", rows(zid) + [("Latitude", f"{p.y:.5f}"),
                                                     ("Longitude", f"{p.x:.5f}")],
                         *tag(zid, "CENTROID", "#de2d26"))).add_to(m)

    # Frame the zoning, not the network: this map is about the zones.
    if len(zones):
        minx, miny, maxx, maxy = zones.total_bounds
        m.fit_bounds([[miny, minx], [maxy, maxx]])

    legend = [("#3182bd", "TAZ zone boundary"), ("#de2d26", "zone centroid")]
    if roads:
        legend.append(("#c4c4c4", "road network"))
    if cut_off:
        legend.append((CUT_OFF_COLOR, "NOT CONNECTED"))
    _add_legend(m, "Zone system", legend)

    info = [f"{len(zones)} zones", f"{len(cents)} centroids",
            f"{zones['area_km2'].sum():,.0f} km&sup2; total (approx.)"]
    if not checked:
        info.append('<span style="color:#555;">centroid connectors not built yet</span>')
    elif cut_off:
        listed = ", ".join(str(z) for z in sorted(cut_off))
        info.append(f'<span style="color:{CUT_OFF_COLOR}; font-weight:bold;">'
                    f'{len(cut_off)} not connected: {listed}</span>')
        print(f"WARNING: {len(cut_off)} zone(s) cannot be served by the network: {listed}.")
        for st, why in CUT.items():
            hit = sorted(z for z in cut_off if state(z) == st)
            if hit:
                print(f"  {why}: {', '.join(str(z) for z in hit)}")
    _add_info(m, "Study area", info)
    return _save_map(m, filename)


def map_zones(filename="1_zones_with_demand.html"):
    """Overview map: the TAZ zones, their centroids, and the road network.

    Hovering a zone or its centroid shows how many trips the baseline demand matrix sends out of
    and into it - the demand side of the model - plus how many of those actually loaded onto the
    network once the baseline is solved.

    Zones that never reached the routing graph are drawn in **orange** and tagged
    ``NOT CONNECTED``: their trips sit in the matrix but the assignment drops them. Better to see
    that on the first map than to wonder later why a corner of the city looks empty.
    """
    zones = read_geo("zones", "zone_id")
    cents = read_geo("nodes", "node_id, is_centroid")
    cents = cents[cents["is_centroid"] == 1]

    conn_info = zone_connection()
    cut_off = {z for z, info in conn_info.items() if zone_is_cut_off(info)}

    def tip(zone_id, connected_tag, connected_color):
        off = int(zone_id) in cut_off
        return (f"Zone {int(zone_id)}", _zone_rows(zone_id, conn_info),
                "NOT CONNECTED" if off else connected_tag,
                CUT_OFF_COLOR if off else connected_color)

    zones["cut_off"] = zones["zone_id"].astype(int).isin(cut_off)
    zones["tip"] = [_tip_html(*tip(z, "TAZ", "#3182bd")) for z in zones["zone_id"]]

    m = _base_map()                               # grey road network + fit to it
    folium.GeoJson(
        zones[["tip", "cut_off", "geometry"]],
        style_function=lambda f: (
            {"color": CUT_OFF_COLOR, "weight": 2, "dashArray": "5,4", "fill": True,
             "fillColor": CUT_OFF_COLOR, "fillOpacity": 0.18}
            if f["properties"]["cut_off"] else
            {"color": "#3182bd", "weight": 1.5, "fill": True,
             "fillColor": "#3182bd", "fillOpacity": 0.06}),
        tooltip=_geojson_tip(),
    ).add_to(m)
    for _, c in cents.iterrows():
        zid, p = int(c.node_id), c.geometry
        off = zid in cut_off
        folium.CircleMarker(
            [p.y, p.x], radius=6 if off else 4,
            color=CUT_OFF_COLOR if off else "#de2d26", fill=True,
            fill_color="#ffffff" if off else "#de2d26",     # hollow = nothing flows here
            fill_opacity=1.0, weight=3 if off else 1,
            tooltip=_tip(*tip(zid, "CENTROID", "#de2d26"))).add_to(m)

    legend = [("#3182bd", "TAZ zone boundary"),
              ("#de2d26", "zone centroid"),
              ("#c4c4c4", "road network")]
    if cut_off:
        legend.append((CUT_OFF_COLOR, "NOT CONNECTED - trips cannot be assigned"))
    _add_legend(m, "Zone system", legend, note="hover a zone for its baseline trips")

    info = [f"{len(zones)} zones", f"{len(cents)} centroids"]
    # Trips actually loaded = the outbound side of every connector, so `assigned` and
    # `unassigned` add up to the matrix total exactly. (Summing the rows and columns of the
    # cut-off zones would double-count the trips between two of them.)
    assigned = sum(i["out"] for i in conn_info.values()) if conn_info else 0.0
    unassigned = max(0.0, float(baseline_demand.sum()) - assigned)
    if cut_off:
        listed = ", ".join(str(z) for z in sorted(cut_off))
        info.append(f'<span style="color:{CUT_OFF_COLOR}; font-weight:bold;">'
                    f'{len(cut_off)} not connected: {listed}</span>')
    if conn_info:
        info.append(f"{assigned:,.0f} veh assigned")
        if unassigned > 0.5:
            info.append(f'<span style="color:#555;">{unassigned:,.0f} veh unassigned</span>')
    if cut_off:
        print(f"WARNING: {len(cut_off)} zone(s) not connected to the car network: {listed}.")
        print(f"  {unassigned:,.0f} trips stay in the matrix but are dropped by the assignment "
              f"(AequilibraE's \"centroids not present in the graph\" warning).")
        # The map says only "not connected"; the cause belongs here, with whoever fixes it.
        for why, zs in (("no centroid connector was built",
                         [z for z in sorted(cut_off) if conn_info[z]["connectors"] == 0]),
                        ("connector exists but the graph never used it",
                         [z for z in sorted(cut_off) if conn_info[z]["connectors"] > 0])):
            if zs:
                print(f"  {why}: {', '.join(str(z) for z in zs)}")
    _add_info(m, "Study area", info)
    return _save_map(m, filename)

print("Reporting helpers defined: compare, map_zone_system, map_zones, map_congestion,\n      map_difference, map_capacity.")


### The zone system (study area)

Before any results, an orientation map: the **Traffic Analysis Zones** (blue) and their
**centroids** (red — the points where trips enter and leave the network), over the road
network. This is the picture of *how the demand is aggregated* — coarser zones mean fewer,
bigger blue cells; finer zones (more of them) load more of the streets.

There are **two** versions of this map, and the difference is what they depend on:

- **`map_zone_system()`** — zones, centroids and (optionally) the road network, drawn from
  *geometry alone*. No demand matrix, no assignment, no results. This is what a GUI can show the
  moment the user draws or imports a zoning, and it is the first cell below.
- **`map_zones()`** — the same picture enriched with the demand matrix and the baseline result:
  trips per zone and the connectivity check. It needs those inputs to exist.

Hover any zone for the trips it sends and receives — the **matrix** rows are the demand input
(row and column sums), the **assigned** rows are what actually loaded onto its connectors. The
assigned figures sit slightly below the matrix, because a trip needs *both* of its zones to be
reachable. **Orange, dashed zones with a hollow centroid are not connected**: the network cannot
serve them, so AequilibraE drops their trips from the assignment and the info box totals the
demand that never travels. The map says just that — **not connected** — because the conclusion
for a reader is the same however it came about: those trips are missing from the results, so that
corner of the picture is not to be trusted.

If you are the one *fixing* it, the cause matters, and the cell output names it:

| Cell output says | What happened | What to do |
|---|---|---|
| `no centroid connector was built` | The zoning never attached the centroid — there was no network node to attach it to, usually because the centroid sits where the downloaded network has nothing. | Widen the OSM extract, or move/redraw that zone. |
| `connector pruned from the graph` | The connector was built, but it hangs off a cul-de-sac, and `prepare_graph` prunes dead ends — taking the connector with them. | Attach the zone to a through street instead of the stub. |
| `connector exists but the graph never used it` | The connector survived, yet no traffic used it: usually an isolated fragment of network, or a zone reachable only *through* another centroid (which `set_blocked_centroid_flows` forbids). | Check what the connector lands on — island, stub, or missing mode `c`. |

`zone_graph_status()` and `zone_connection()` return the same information programmatically, and
**none of it needs the demand matrix**: whether a zone can be served is decided by
`prepare_graph()`, which reads the network and the centroids only. Map 1's check happens to read
assigned volumes because, once a baseline exists, the answer is already sitting in the result.


In [ ]:
# Geometry only: this one needs nothing but the zoning - no demand, no assignment.
map_zone_system("0_zones_and_centroids.html")

In [ ]:
map_zones("1_zones_with_demand.html")

### The baseline congestion map

Blue is free-flowing, red is over capacity; thicker lines carry more cars.


In [ ]:
map_congestion(baseline, "2_baseline_congestion.html")

# Part C — The four scenarios

Each scenario is a small function returning a `ScenarioResult`, so you *run it, then
`compare` and map it against the cached `baseline`*. In a GUI, each of these is one button.

| Scenario | What changes | Touches the database? |
|---|---|---|
| Close a road | links removed from routing | No (in-memory) |
| Change capacity / speed | `capacity` / `travel_time` of some links | No (in-memory) |
| Change demand | the O-D matrix | No (in-memory) |
| Add a new road | a link is inserted, then removed | **Briefly** — undone afterwards |

The first three leave the network on disk untouched, so they are perfectly reversible and
can be run in any order. Adding a road writes a real link to the database, but we **delete it
again** as soon as the scenario has been evaluated, so the network returns to its baseline
state — you can try one alternative road after another, and the other scenarios are never
affected.


## Scenario 1 — Close a road

Removing links from routing is a one-liner: `build_graph(exclude=link_ids)`. To pick a
target we use `pick_corridor()`, a small helper that finds a target arterial and grows a
*contiguous* section along it (one coherent stretch, not scattered bits). It can rank
arterials three ways (`CORRIDOR_METRICS`):

- **`by="volume"`** — the busiest artery. Best for a "major road is blocked" story, so that is
  what this scenario uses.
- **`by="delay"`** — where the most vehicle-minutes are actually lost to queueing. This is the
  default, and the right target for the relief scenarios (widening, new road) coming next:
  adding capacity only pays where time is being lost.
- **`by="voc"`** — the worst *bottleneck* (highest volume/capacity ratio). Good for spotting a
  single pinch point, but a poor guide to where to spend money — the highest V/C is often a
  low-capacity residential street where extra lanes buy almost nothing. Scenario 2 returns to
  this.


In [ ]:
def _arterials():
    """Baseline results joined to link attributes: named arterials only."""
    base = baseline.results.reset_index().merge(link_attr, on="link_id", how="left")
    return base[base.link_type.isin(["motorway", "trunk", "primary", "secondary"])
                & base.name.notna()].copy()


def _delay_minutes(df):
    """Vehicle-minutes LOST to congestion on each link, both directions.

    The assignment reports `Delay_factor` = congested time / free-flow time, so the part of the
    congested time that is *queueing* is `congested * (1 - 1/factor)`. Multiplied by the flow it
    becomes vehicle-minutes of delay - the quantity a widening is supposed to buy back, and a far
    better target than the highest V/C (which can be a quiet residential street with a low
    capacity, where widening buys almost nothing).
    """
    total = 0.0
    for d in ("ab", "ba"):
        flow = df[f"car_{d}"].fillna(0.0)
        time = df[f"Congested_Time_{d.upper()}"].fillna(0.0)
        factor = df[f"Delay_factor_{d.upper()}"].replace(0.0, np.nan)
        share = (1.0 - 1.0 / factor).fillna(0.0).clip(lower=0.0)   # 0 when free-flowing
        total = total + flow * time * share
    return total


# How to rank arterials. Each entry is (column, how to aggregate along the street).
CORRIDOR_METRICS = {
    "delay":  ("delay_min", "sum"),    # where the city loses the most time  <- best for widening
    "volume": ("car_tot", "sum"),      # the busiest artery
    "voc":    ("VOC_max", "max"),      # the single worst bottleneck link
}


def _street_ranking(by):
    art = _arterials()
    art["delay_min"] = _delay_minutes(art)
    col, how = CORRIDOR_METRICS[by]
    return art, col, art.groupby("name")[col].agg(how).sort_values(ascending=False)


def rank_corridors(by="delay", top=6):
    """The worst arterials by `by` - a table you can read before choosing what to build.

    Ratios keep two decimals; vehicle-minutes and vehicles are whole numbers. (Rounding V/C to
    whole numbers would show half the arterials tied at "1", which is exactly the kind of
    false tie that makes a bad target look reasonable.)
    """
    _, col, order = _street_ranking(by)
    label = {"delay": "delay (veh-min lost)", "volume": "volume (veh)",
             "voc": "worst V/C"}.get(by, by)
    return order.head(top).round(2 if by == "voc" else 0).rename(label).reset_index()


def _grow_section(corr, metric, section_links):
    """Grow a *contiguous* run of links along one street, starting from its worst link."""
    start = corr.sort_values(metric, ascending=False).iloc[0]
    picked = {int(start.link_id)}
    nodes = {int(start.a_node), int(start.b_node)}
    while len(picked) < section_links:
        nbrs = corr[(~corr.link_id.isin(picked))
                    & (corr.a_node.isin(nodes) | corr.b_node.isin(nodes))]
        if nbrs.empty:
            break
        nxt = nbrs.sort_values(metric, ascending=False).iloc[0]
        picked.add(int(nxt.link_id))
        nodes |= {int(nxt.a_node), int(nxt.b_node)}
    return list(picked)


def pick_corridor(by="voc", section_links=14, rank=0):
    """Return (street_name, [link_ids]) for a contiguous stretch of a target arterial.

    by="delay":  where the most vehicle-minutes are lost to congestion (best widening target).
    by="volume": the busiest artery (highest total traffic).
    by="voc":    the worst bottleneck (highest volume/capacity ratio).
    rank=1, 2, ... walks down the ranking instead of taking the worst.
    """
    art, col, order = _street_ranking(by)
    road = order.index[rank]
    return road, _grow_section(art[art.name == road], col, section_links)


def pick_corridors(n=3, by="delay", section_links=20):
    """The `n` worst corridors, each as (street_name, [link_ids]): a *package* of works.

    One widening is easy for the network to absorb - drivers refill the freed space. Treating
    several corridors as one project is both more realistic (that is how road programmes are
    funded) and far more visible on the map.
    """
    return [pick_corridor(by=by, section_links=section_links, rank=r) for r in range(n)]


def scenario_close(link_ids, label="Close road"):
    return solve_car(build_graph(exclude=link_ids), make_demand(baseline_demand), label)

In [ ]:
road_name, closed_links = pick_corridor(by="volume", section_links=14)
print(f"Closing a {len(closed_links)}-link section of the busiest artery: '{road_name}'.")

closure = scenario_close(closed_links, label="Road closed")
changes_close = compare(baseline, closure)

print("\nStreets absorbing the most rerouted traffic:")
print(changes_close.sort_values("change", ascending=False)
      .head(6)[["name", "link_type", "vol_base", "vol_scen", "change"]].to_string(index=False))

map_difference(changes_close, closed=closed_links,
               base_time=baseline.system_time, scen_time=closure.system_time,
               filename="3_close_road.html")

## Scenario 2 — Change capacity or speed

Instead of closing a road we can make it *bigger* or *smaller*. `scenario_capacity`
multiplies the `capacity` of the chosen links (e.g. `1.5` = add a lane, `0.5` = a lane
closed) and/or sets a new free-flow `speed` (which recomputes travel time). This all
happens on the in-memory graph, so the baseline is untouched.

Below we widen a **package of corridors** rather than a single link, and we choose them by
**delay** — the vehicle-minutes actually lost to queueing on each street
(`rank_corridors(by="delay")`). That target matters: the highest *V/C* can easily be a quiet
residential street whose capacity is low, where extra lanes buy almost nothing. Ranking by delay
finds the streets where the city is really losing time, so the money goes where it can be won
back. `N_CORRIDORS`, `CAPACITY_FACTOR` and `SECTION_LINKS` are the knobs — set `N_CORRIDORS = 1`
to see how much less a single widening achieves.

Mind the units when reading the counts: OSM splits a street at **every intersection**, so one
avenue is dozens of links (and a dual carriageway is two links side by side). Widening 3
corridors with `SECTION_LINKS = 20` therefore edits **60 links** — three roads on the map. The
info panel reports both, plus the length in km, so the two never look contradictory.

Watch the two numbers. The **city-wide** total moves only modestly even for a programme this
size, while **locally** the corridors clear. That gap is the important lesson: because the
network re-optimises, freed-up space is quickly refilled by drivers who reroute onto it. The
value of the project shows up on the **map**, not in the headline number — which is also why a
package of works is worth mapping: one widening is a dot, several are a pattern.

The map for this scenario is a **congestion-change map (V/C)**, *not* a volume map: **blue**
where congestion eased (improved), **red** where it worsened, with the widened links haloed
in **gold**. A volume map would actively mislead here — a widened road usually attracts
*more* cars, so it would paint the improved corridor red ("more traffic") even though it is
now less congested. Congestion change is what tells you whether the situation got better.


In [ ]:
def scenario_capacity(link_ids, capacity_factor=1.0, new_speed=None, label="Capacity change"):
    ids = set(int(i) for i in link_ids)

    def edit(df):
        mask = df["link_id"].isin(ids)
        if capacity_factor != 1.0:
            df.loc[mask, "capacity"] = df.loc[mask, "capacity"] * capacity_factor
        if new_speed is not None:                      # recompute free-flow time from new speed
            df.loc[mask, "travel_time"] = (df.loc[mask, "distance"] / 1000.0) / new_speed * 60.0

    return solve_car(build_graph(edit=edit), make_demand(baseline_demand), label)

In [ ]:
# A PACKAGE of works, not one link: widen the corridors where the city loses the most time.
N_CORRIDORS = 3          # how many arterials to widen together
CAPACITY_FACTOR = 2.0    # 2.0 = double the capacity (1.5 would be one extra lane)
SECTION_LINKS = 20       # length of the widened stretch on each corridor

print("Arterials ranked by vehicle-minutes lost to congestion (the widening targets):")
print(rank_corridors(by="delay", top=6).to_string(index=False))

works = pick_corridors(n=N_CORRIDORS, by="delay", section_links=SECTION_LINKS)
widened_links = [lid for _, ids in works for lid in ids]
print(f"\nWidening {len(works)} corridors ({len(widened_links)} links) "
      f"to {CAPACITY_FACTOR:g}x capacity:")
for name, ids in works:
    print(f"  {name}  ({len(ids)} links)")

widen = scenario_capacity(widened_links, capacity_factor=CAPACITY_FACTOR,
                          label=f"{N_CORRIDORS} corridors x{CAPACITY_FACTOR:g}")
changes_cap = compare(baseline, widen)

print("\nStreets most relieved by the extra capacity (biggest volume drop):")
print(changes_cap.sort_values("change").head(6)
      [["name", "link_type", "vol_base", "vol_scen", "change", "change_pct"]]
      .round(0).to_string(index=False))

# Congestion-change map (V/C), NOT a volume map: blue = less congested, gold = widened links.
map_capacity(baseline, widen, widened_links, filename="4_capacity_change.html")


## Scenario 3 — Change demand (a growing travel corridor)

Here the network stays fixed and the **trips** change. Rather than a bland city-wide
increase — which just makes everything uniformly redder and tells you nothing — we grow
demand along **one origin-to-destination axis**: think of a booming suburb that
increasingly commutes downtown. We inject a block of new trips from an **origin zone** to a
**destination zone** and watch which roads carry them and where they congest.

This is also the cleanest demand edit to drive from a GUI: the user clicks an **origin
zone**, clicks a **destination zone**, and sets an **amount** — no need to touch the full
matrix, and no assumption about how trips spread, because both ends are named.
`scenario_demand(add=N, origins=[A], destinations=[B])` does exactly that. (The same
function still does city-wide growth — call it with just a `factor` and no zones.)


In [ ]:
def scenario_demand(factor=1.0, add=0.0, origins=None, destinations=None, label="Demand change"):
    """Edit demand for an origin->destination block, then re-solve.

    origins / destinations: zone ids (1..num_zones), or None meaning "all zones".
    Each cell in the selected block becomes  cell * factor + add:
      - use `factor` to grow existing trips (e.g. factor=1.3 for +30%),
      - use `add` to inject new trips per O-D cell (e.g. a new commuter flow A->B).
    """
    demand = baseline_demand.copy()
    o_idx = np.arange(num_zones) if origins is None else np.array([z - 1 for z in origins])
    d_idx = np.arange(num_zones) if destinations is None else np.array([z - 1 for z in destinations])
    block = np.ix_(o_idx, d_idx)
    demand[block] = demand[block] * factor + add
    np.fill_diagonal(demand, 0)                  # never create intra-zonal trips
    return solve_car(build_graph(), make_demand(demand), label)

In [ ]:
# A growing commuter flow between two zones (a GUI would let the user click these on the map).
ORIGIN_ZONE, DEST_ZONE = 15, 58
NEW_TRIPS = 1500                        # extra trips/hour from ORIGIN_ZONE to DEST_ZONE

growth = scenario_demand(add=NEW_TRIPS, origins=[ORIGIN_ZONE], destinations=[DEST_ZONE],
                         label=f"+{NEW_TRIPS} trips {ORIGIN_ZONE}->{DEST_ZONE}")
changes_dem = compare(baseline, growth)

print(f"\nAdded {NEW_TRIPS} trips/hour from zone {ORIGIN_ZONE} to zone {DEST_ZONE}.")
print("Streets carrying the most of the new corridor traffic:")
print(changes_dem.sort_values("change", ascending=False)
      .head(6)[["name", "link_type", "vol_base", "vol_scen", "change"]].to_string(index=False))

# The two markers carry the SAME added flow, so each says where it goes ("Sent to" /
# "Received from") rather than repeating a bare "+1,500": otherwise the map cannot tell you
# whether 1,500 or 3,000 trips were added, nor in which direction. The info panel states the
# flow once, which is what settles the total.
_flow = f"zone {ORIGIN_ZONE} &rarr; zone {DEST_ZONE}"
map_difference(changes_dem, filename="5_demand_change.html",
               mark_zones=[(ORIGIN_ZONE, "ORIGIN",
                            [("Added demand", f"+{NEW_TRIPS:,} veh"),
                             ("Sent to", f"zone {DEST_ZONE}")]),
                           (DEST_ZONE, "DESTINATION",
                            [("Added demand", f"+{NEW_TRIPS:,} veh"),
                             ("Received from", f"zone {ORIGIN_ZONE}")])],
               info_extra=[f'<div style="margin-top:5px; border-top:1px solid #ddd; '
                           f'padding-top:5px;">Demand added: <b>+{NEW_TRIPS:,} veh</b>, '
                           f'{_flow}</div>'],
               base_time=baseline.system_time, scen_time=growth.system_time)

## Scenario 4 — Add a new road

Adding a road is *geometry-driven*: we create a link from a `LineString` whose two
endpoints sit **exactly** on existing network nodes. Database triggers then snap it in
place and fill `a_node`, `b_node` and `distance`. (If an endpoint misses an existing node,
a disconnected node is created and the road carries no traffic — so matching coordinates is
what makes the road actually connect.)

Unlike the other three, this **writes to the database** — but only briefly: after we solve
and map the scenario we **delete the link again** (`remove_link`), so the network returns to
baseline and you can try one alternative road after another. In a GUI, that deletion is the
"discard this idea / try another" action.

**Where you put it matters more than what it is.** A new road only carries traffic if it sits
on an axis people are already trying to travel, so the question is where the traffic *is*. Note
where that answer does **not** live: the demand matrix. Ours is near-uniform — every O-D pair
gets about the same handful of trips — yet the streets are very unevenly loaded, because the
network makes thousands of paths overlap on a few of them. Concentration is created by the
*assignment*, so that is what we read.

`add_express_link` therefore:

1. scores every **zone** by the baseline traffic on the streets inside it (`zone_traffic()`);
2. takes the pair of busy zones with the highest *geometric* mean traffic — so **both** ends
   are busy, not just one — subject to being at least `min_separation_km` apart (4 km default);
3. touches down on the busiest **node** in each of the two zones, because that is where the
   flows already pass, and a link that misses them is a link nobody can reach.

Since the busiest zones sit in the middle of the study area, the winning pair straddles the
congested core — the axis most trips are trying to cross — and the road gets used.

> **Why not simply parallel the busiest street?** That was the obvious first idea, and it fails:
> its endpoints land at the *extremities* of the busiest corridor, out at the edge of the city
> where few trips start or end. The link is fast, but almost nobody is in a position to use it.
> A road is only as useful as the two places it joins. In the run this notebook was written
> from, the two placements were not close: **53 vehicles** on an 18 km road along the busiest
> corridor, against **~3,900 vehicles** on a 4 km road between the two busiest zones — same
> speed, same capacity, same demand. Position is the whole story.

`min_separation_km` is the knob: raise it for a longer, more ambitious road, lower it for a
short urban connector.

The demand matrix is **untouched** — this scenario changes the network only, so every trip on
the map is a baseline trip that chose a different route. (That is also why the map carries no
purple zone markers: those mean "origin / destination of *added* demand", which belongs to
scenario 3 alone.) Some drivers shift onto the new link; the difference map shows who. As with
the widening, expect *local* redistribution to be the story — a single new link nudges the
citywide total only a little.


In [ ]:
def _km(p, q):
    """Straight-line distance in km between two lon/lat points (fine over a city, near 0 lat)."""
    mid_lat = np.radians((p.y + q.y) / 2)
    return float(np.hypot((q.x - p.x) * 111.32 * np.cos(mid_lat), (q.y - p.y) * 110.57))


def zone_traffic():
    """Score every zone by the baseline traffic on the streets inside it.

    Where should a new road go? Not where the *demand matrix* is largest - our synthetic
    demand is near-uniform, every O-D pair gets about the same handful of trips. Traffic
    concentrates because of the *network*: many paths overlap on a few streets. So we read the
    concentration off the baseline assignment instead.

    Returns one row per zone with its total traffic and its busiest network node (the point a
    new road should touch down on, because that is where the flows already pass).
    """
    res = baseline.results.reset_index().merge(link_attr, on="link_id", how="left")
    res = res[res["link_type"] != "centroid_connector"]

    # Traffic through a node = the volume on the links that meet there.
    node_vol = (pd.concat([res[["a_node", "car_tot"]].rename(columns={"a_node": "node_id"}),
                           res[["b_node", "car_tot"]].rename(columns={"b_node": "node_id"})])
                .groupby("node_id", as_index=False)["car_tot"].sum()
                .rename(columns={"car_tot": "node_vol"}))

    nodes = read_geo("nodes", "node_id, is_centroid")
    nodes = nodes[nodes["is_centroid"] == 0].merge(node_vol, on="node_id", how="inner")
    zones = read_geo("zones", "zone_id")
    inside = gpd.sjoin(nodes, zones[["zone_id", "geometry"]], predicate="within", how="inner")

    total = inside.groupby("zone_id", as_index=False)["node_vol"].sum().rename(
        columns={"node_vol": "traffic"})
    busiest = inside.sort_values("node_vol", ascending=False).drop_duplicates("zone_id")
    out = total.merge(busiest[["zone_id", "node_id", "node_vol", "geometry"]], on="zone_id")
    return (gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")
            .sort_values("traffic", ascending=False).reset_index(drop=True))


def busiest_zone_pair(min_separation_km=4.0, top_zones=10, verbose=True):
    """Pick the two heavily-trafficked zones a new road should connect.

    Among the `top_zones` busiest zones we take the pair with the highest **geometric mean** of
    traffic - so *both* ends must be busy, not just one - subject to being at least
    `min_separation_km` apart. The separation matters twice over: it stops us "connecting" two
    nodes of the same junction, and it is what makes the new road a genuine shortcut. Because
    the busiest zones sit in the middle of the study area, the winning pair straddles the core,
    which is exactly the axis most trips are already trying to cross.

    Returns the two rows of `zone_traffic()` (zone id, traffic, busiest node + geometry).
    """
    zt = zone_traffic().head(top_zones)
    best = None
    for i in range(len(zt)):
        for j in range(i + 1, len(zt)):
            a, b = zt.loc[i], zt.loc[j]
            km = _km(a.geometry, b.geometry)
            if km < min_separation_km:
                continue                      # too close to be a corridor
            score = float(np.sqrt(a["traffic"] * b["traffic"]))
            if best is None or score > best[0]:
                best = (score, km, a, b)
    if best is None:
        raise RuntimeError(f"None of the {top_zones} busiest zones are {min_separation_km} km "
                           "apart. Lower min_separation_km or raise top_zones.")
    _, km, a, b = best
    if verbose:
        print(f"New road axis: zone {int(a.zone_id)} <-> zone {int(b.zone_id)}, "
              f"{km:.1f} km apart (of the {top_zones} busiest zones).")
        for z in (a, b):
            print(f"  zone {int(z.zone_id):>3}: {z['traffic']:>9,.0f} veh on its streets, "
                  f"touching down on node {int(z.node_id)} ({z['node_vol']:,.0f} veh)")
    return a, b


def add_express_link(speed=80, capacity=3000, link_type="trunk", min_separation_km=4.0):
    """Insert a new express link between the two busiest zones.

    Returns (link_id, info) where `info` records the axis chosen, so the caller can report and
    map it. The endpoints are the busiest existing *network node* of each zone: the geometry
    lands exactly on them, which is what lets the database triggers snap the link into the
    network (miss a node and the road connects to nothing and carries no traffic).
    """
    a, b = busiest_zone_pair(min_separation_km=min_separation_km)
    pa, pb = a.geometry, b.geometry
    a_id, b_id = int(a.node_id), int(b.node_id)
    name = f"Nueva Via Expresa (zona {int(a.zone_id)} - zona {int(b.zone_id)})"

    # refresh_fields() re-reads max(link_id) before we create the link: the bulk connector
    # step used raw SQL, so the cached id counter is stale and new() could reuse an id.
    links_table = project.network.links
    links_table.refresh_fields()
    link = links_table.new()
    link.geometry = LineString([(pa.x, pa.y), (pb.x, pb.y)])
    link.modes = "c"
    link.link_type = link_type
    link.name = name
    link.direction = 0
    link.speed_ab = link.speed_ba = speed
    link.capacity_ab = link.capacity_ba = capacity
    new_id = link.link_id
    link.save()

    with project.db_connection as conn:
        conn.execute("UPDATE links SET travel_time_ab=(distance/1000.0)/speed_ab*60.0, "
                     "travel_time_ba=(distance/1000.0)/speed_ba*60.0 WHERE link_id=?", (new_id,))
        conn.commit()
        chk = pd.read_sql(f"SELECT a_node, b_node, distance FROM links WHERE link_id={new_id}",
                          conn)
    assert {int(chk.a_node[0]), int(chk.b_node[0])} == {a_id, b_id}, \
        "Endpoints did not snap to the intended nodes."

    length_km = float(chk.distance[0]) / 1000.0
    info = {"name": name, "zone_a": int(a.zone_id), "zone_b": int(b.zone_id),
            "traffic_a": float(a["traffic"]), "traffic_b": float(b["traffic"]),
            "node_a": a_id, "node_b": b_id, "length_km": length_km}
    print(f"Added link {new_id}: {name}")
    print(f"  {length_km:.1f} km between nodes {a_id} and {b_id}, "
          f"{speed} km/h, capacity {capacity:,} veh/h "
          f"(vs {60 * length_km / speed:.0f} min at free flow).")
    return new_id, info


def scenario_new_road(speed=80, capacity=3000, min_separation_km=4.0, label="New express road"):
    new_id, info = add_express_link(speed=speed, capacity=capacity,
                                    min_separation_km=min_separation_km)
    result = solve_car(build_graph(), make_demand(baseline_demand), label)
    return result, new_id, info


def remove_link(link_id):
    """Delete a link added by add_express_link, restoring the baseline network.

    The new link snapped to two *existing* nodes (add_express_link asserts this), so only the
    link row is removed - no nodes are orphaned. This makes the "add a road" scenario fully
    repeatable: undo it and try a different road. refresh_fields() re-reads the id counter so
    the next add_express_link picks a fresh id.
    """
    with project.db_connection as conn:
        conn.execute("DELETE FROM links WHERE link_id=?", (int(link_id),))
        conn.commit()
    project.network.links.refresh_fields()


In [ ]:
newroad, new_link_id, road_info = scenario_new_road(label="New express road")

# Re-read geometry AND link attributes so the just-added link is drawable and named, then
# compare. `compare` returns the new link too (baseline volume 0), which is how the map
# tooltip can report how many vehicles it carries.
links_geo = read_geo("links", "link_id, name, link_type")
with project.db_connection as conn:
    link_attr = pd.read_sql(
        "SELECT link_id, name, link_type, a_node, b_node, distance FROM links", conn)
changes_new = compare(baseline, newroad)

_new = newroad.results.reset_index().query("link_id == @new_link_id")
new_vol, new_voc = float(_new.car_tot.iloc[0]), float(_new.VOC_max.iloc[0])
print(f"\nThe new express link carries {new_vol:,.0f} vehicles "
      f"({new_voc:.0%} of its capacity).")
print(f"It connects zone {road_info['zone_a']} to zone {road_info['zone_b']}, "
      f"{road_info['length_km']:.1f} km, on the same demand as the baseline.")
print("\nStreets most relieved by the new road (biggest drop):")
print(changes_new.sort_values("change")
      .head(6)[["name", "link_type", "vol_base", "vol_scen", "change", "change_pct"]]
      .round(0).to_string(index=False))

# No `mark_zones` here: the purple zone markers mean "origin / destination of *added
# demand*", and this scenario changes only the network - the demand is the baseline matrix.
new_road_map = map_difference(changes_new, new=[new_link_id],
                              base_time=baseline.system_time, scen_time=newroad.system_time,
                              filename="6_new_road.html")

# Undo the new link so the network returns to baseline: you can then try a different road
# (change the target or attributes and re-run) and the other scenarios stay unaffected.
# In a GUI, this deletion is the "discard scenario / try another" action.
remove_link(new_link_id)
links_geo = read_geo("links", "link_id, name, link_type")   # refresh geometry (link removed)
print(f"\nRemoved link {new_link_id}; network restored to baseline.")

new_road_map   # render inline (the map was saved before the link was removed)

# From notebook to GUI

Every scenario above is the same three moves: **edit an input, call `solve_car`, then
`compare`.** That is exactly the shape a graphical interface needs:

- **Baseline is solved once and cached** (`baseline`). The GUI does this on model load.
- **Each `scenario_*` function is one button.** Its arguments are the controls:
  a list of links to close, a capacity slider, an origin + destination + amount for demand,
  a drawn road.
- **`compare` and the `map_*` functions are the results views**, shared by every scenario —
  congestion uses one colour scale, difference maps another, and `map_capacity` reports the
  change in V/C, which is the honest view of a widening.
- **The first three scenarios never touch the database**, so they are instantly reversible;
  *add a road* writes to disk but is **undone straight after** (`remove_link`), so every
  scenario leaves the baseline network intact and can be retried with different inputs.

Two data caveats before trusting the numbers on a real study: the O-D matrix here is
**synthetic**, and the link capacities are **illustrative defaults** — both must be
replaced with calibrated data for the metrics to be meaningful.
